# 06 — Incremental batch demand and performance forecasts

Run after the new notebook 05. Reads completed similarity-generation records, prepares shared Sales/Rental/Eilers data once for pending experiments, and saves immutable forecast runs with durable progress. No scheduler or interactive choice is required.

## Setup

1. Run the updated 05 using your existing YAML configuration.
2. Run all cells in 06. It selects completed generations using the updated 05 rules and appends new versioned forecasts. Previous batch-format results remain available.
3. Run one manual writer at a time. A retry assumes any previous writer has stopped.

## YAML and source revisions

`forecast.settings` contains distance cutoff, minimum neighbors, horizon, slot-count filter, performance metrics, and the rental value/labels. The rental definition retains `source_row_count` as monthly rental records. `forecast.selection.similarity_experiment_ids` optionally restricts discovery; `[]` uses the matching completed 05 generation for each enabled named experiment.

`forecast.data_revision` identifies the Sales/Rental/Eilers Delta versions shared with neighbor selection in 05. Later experiments reuse that snapshot for fair comparison. Change this revision explicitly to forecast from refreshed sources. An explicit source-version conflict within an existing revision is rejected. The AI snapshot remains governed independently by 05's `data.revision`.

Forecast identity includes the saved 05 generation identity/result hash, forecast settings, implementation version, and pinned forecast data revision/snapshots. The name, rank weights, and correction choice belong to the forecast identity. A rerun with the same inputs reuses the saved forecast. Completed work is verified/skipped across full reruns and cluster restarts. Increment `FORECAST_IMPLEMENTATION_VERSION` when intentionally changing computational behavior.

## Inference and calendar rules

- Every valid target is retained for Sales, Rental, and both owned/leased Eilers populations, even when the target has no actuals/source mappings.
- Use 05's explicit `prediction_date`, falling back to `game_release_date`. The target calendar starts at that prediction month; target observation dates are not used to construct predictions. Missing prediction dates remain visible as null forecasts with a reason.
- Notebook 06 consumes the final ordered sales, rental, and Eilers lists saved by 05. It loads generations whose selection settings and source snapshots match the current YAML. To change a selection setting, run 05 first.
- For commercial neighbors, `forecast_neighbors_kept` shows the selection threshold date from 05, the required minimum, and the observed month count from the shared historical snapshot. The defaults are six sales months and nine rental months.

- Only neighbor source months strictly before the target prediction month are available. Apply that cutoff before finding the neighbor's start/end or filling gaps. Relative neighbor month 1 is its first historically observed source month.
- Sales: internal absent monthly values become zero, then cumulative sum. Rental: internal absent levels forward-fill. Eilers: metric-specific weighted statistics forward-fill internally; leading metric nulls remain. No curve is extended beyond the last available as-of source observation.
- Eilers weights use positive `no_of_slots` and nonnull metric denominators, after averaging duplicate game/month/slot buckets. `slot_counts: []` accepts every positive bucket.
- Bands describe neighbor dispersion, not calibrated prediction intervals. Changes in contributing neighbors can cause a cumulative-sales average to decline; this is flagged in diagnostics and is not silently clipped.

## Stage 4 demand curves

Each enabled entry in `experiments` produces a forecast with its YAML name. Its `forecast.rank_weights` apply to the available neighbors ranked by demand in each relative month. Sales use cumulative sales and rentals use monthly rental volume. The first number weights the highest value. Surplus weights are ignored if fewer neighbors contribute, and additional neighbors receive the last configured weight. Tied values give the same result in any order. The monthly benchmark and both dispersion bands use these weights. Eilers indices use their existing slot weights.

`forecast_value` is the resulting monthly demand benchmark. Each run fits three origin-constrained curves to that benchmark for each target and revenue type. `forecast_linear` uses a × month, `forecast_poly2` uses a × month + b × month², and `forecast_log` uses a × log(1 + month). Linear and log require two benchmark months. Quadratic requires three. The fit uses the month indices with finite benchmarks and evaluates across the configured horizon. Negative fitted minima are shifted to zero. Values after the last supported month are extrapolations. A quadratic can turn down even for cumulative sales.

The run diagnostics contain `demand_curve_fits` with the supported month count, last supported month, and best model by centered R² for each game/channel. A constant benchmark has undefined R². The baseline, bands, and three fitted values can be reviewed independently. Eilers performance indices retain their separate output.

## Predictions and actuals

Target actuals are not attached or used to determine eligibility/calendar. Compatibility actual columns are null, observed/imputed flags false, and `actual_status` is `not_attached`. Set `train_bias_correction: true` on a forecast experiment to have 07 learn and save additional demand curves from training games. Use companion 07 to load actuals independently, enforce per-channel/metric ground-truth eligibility, and version evaluation cohorts without regenerating predictions. Game charts live in `Review_Forecasts`.

## Results, retries, and monitoring

All tables live under the configured schema:

| Table | Content |
| --- | --- |
| `demand_forecast_results` | Sales/Rental predictions, bands, coverage, origin and lineage |
| `performance_forecast_results` | Owned/leased Eilers predictions and per-metric coverage |
| `forecast_neighbors_kept` | Exact retained neighbors and as-of source availability |
| `forecast_runs` | Durable processing/tracking status, source pins, immutable signatures and completion |
| `forecast_run_attempts` | Persistent attempt/error/interruption history |

A forecast becomes `complete` only after all three output tables have been saved and verified. Failed/partial attempts reuse their forecast ID and source snapshots. Independent failures do not stop other experiments; the batch raises after displaying its summary. No pending work is a successful no-op.

MLflow settings/coverage under `/Shared/AI_Feature_Extraction_Forecasts` are tracked separately from processing success. A tracking outage does not invalidate verified Delta output; rerunning repairs logging/lineage without regenerating curves. Integrity fingerprints preserve exact nonfloating values, array order and canonical map ordering; floating fields use 12 significant digits in fingerprints to tolerate irrelevant reduction-order noise. Saved predictions retain full precision.

The last cell shows processing/tracking status, attempts, duration, source versions, coverage, and diagnostic reasons. These counts measure forecast production; MAE/RMSE/bias and actual-based train/validation/test eligibility belong to updated 07.

Local behavioral tests and code-cell compilation are included in validation. Live Spark/Delta/MLflow integration still requires Databricks.


In [0]:
# Only notebook input. All forecast settings come from this YAML.
CONFIG_PATH = "configs/experiments.yaml"


## Shared YAML validation

Definitions only; no similarity generation or table writes occur here.


In [0]:
"""Validated YAML configuration and semantic work identities for notebook 05."""
import copy
import hashlib
import json
import math
import re
from datetime import date, datetime
from numbers import Integral, Real
from pathlib import Path

SIMILARITY_IMPLEMENTATION_VERSION = "05_history_at_selection_v3"
WEIGHT_GROUPS = ("mechanics", "themes", "reels", "persistence", "ways", "months_since_release")
PARENT_PROCESSES = {"p1_parent_games_only", "p2_current", "p3_include_parent"}


def config_json(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False,
                      default=lambda x: x.isoformat() if isinstance(x, (date, datetime)) else str(x))


def config_sha256(value):
    return hashlib.sha256(config_json(value).encode("utf-8")).hexdigest()


def quoted_table(name):
    return ".".join("`" + p.replace("`", "``") + "`" for p in name.split("."))


def _keys(value, allowed, where, required=()):
    if not isinstance(value, dict):
        raise ValueError(f"{where} must be a mapping.")
    unknown, missing = set(value) - set(allowed), set(required) - set(value)
    if unknown or missing:
        raise ValueError(f"{where}: unknown keys={sorted(unknown)}, missing keys={sorted(missing)}")


def _text(value, where):
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"{where} must be a nonempty string.")
    return value.strip()


def _integer(value, where, minimum=0):
    if isinstance(value, bool) or not isinstance(value, Integral) or value < minimum:
        raise ValueError(f"{where} must be an integer >= {minimum}.")
    return int(value)


def _date(value, where):
    if value is None:
        return None
    if isinstance(value, datetime):
        value = value.date()
    if isinstance(value, date):
        return value.isoformat()
    try:
        return date.fromisoformat(str(value)).isoformat()
    except ValueError as exc:
        raise ValueError(f"{where} must be YYYY-MM-DD or null.") from exc


def _table(value, where):
    value = _text(value, where)
    if len(value.split(".")) != 3 or not all(re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", x) for x in value.split(".")):
        raise ValueError(f"{where} must be a three-part catalog.schema.table name.")
    return value


def validate_similarity_settings(settings):
    allowed = {"raw_weights", "distance_method", "parent_game_process", "neighbor_limits",
               "neighbor_released_after", "minimum_age_gap_months"}
    _keys(settings, allowed, "similarity settings", allowed)
    settings = copy.deepcopy(settings)
    settings["distance_method"] = _text(settings["distance_method"], "distance_method").lower()
    if settings["distance_method"] not in {"cosine", "euclidean"}:
        raise ValueError("distance_method must be cosine or euclidean.")
    if settings["parent_game_process"] not in PARENT_PROCESSES:
        raise ValueError(f"Unknown parent_game_process: {settings['parent_game_process']!r}")
    _keys(settings["raw_weights"], WEIGHT_GROUPS, "raw_weights", WEIGHT_GROUPS)
    for key, value in settings["raw_weights"].items():
        if isinstance(value, bool) or not isinstance(value, Real) or not math.isfinite(float(value)) or value < 0:
            raise ValueError(f"raw_weights.{key} must be a finite nonnegative number.")
        settings["raw_weights"][key] = float(value)
    if not any(settings["raw_weights"].values()):
        raise ValueError("At least one feature-group weight must be positive.")
    _keys(settings["neighbor_limits"], {"sales", "rental", "eilers"}, "neighbor_limits", {"sales", "rental", "eilers"})
    settings["neighbor_limits"] = {k: _integer(v, f"neighbor_limits.{k}") for k, v in settings["neighbor_limits"].items()}
    settings["minimum_age_gap_months"] = _integer(settings["minimum_age_gap_months"], "minimum_age_gap_months")
    settings["neighbor_released_after"] = _date(settings["neighbor_released_after"], "neighbor_released_after")
    if settings["neighbor_released_after"] is None:
        raise ValueError("neighbor_released_after must be a date.")
    return settings


def named_forecast_settings(item, similarity):
    """Weights are ordered by historical demand in each month, highest first."""
    name = item["name"]
    spec = item.get("forecast", {})
    _keys(spec, {"rank_weights", "train_bias_correction"}, f"{name}.forecast")
    weights = spec.get("rank_weights", [1.0])
    if not isinstance(weights, list) or not weights:
        raise ValueError(f"{name}.forecast.rank_weights must be a nonempty list.")
    if any(isinstance(w, bool) or not isinstance(w, Real) or not math.isfinite(w) or w <= 0 for w in weights):
        raise ValueError(f"{name}.forecast.rank_weights must contain positive finite numbers.")
    if any(a < b for a, b in zip(weights, weights[1:])):
        raise ValueError(f"{name}.forecast.rank_weights must be ordered highest weight first.")
    if not math.isfinite(sum(weights)):
        raise ValueError(f"{name}.forecast.rank_weights sum must be finite.")
    correct = spec.get("train_bias_correction", False)
    if not isinstance(correct, bool):
        raise ValueError(f"{name}.forecast.train_bias_correction must be true or false.")
    return {"name": name, "rank_weights": [float(w) for w in weights],
            "train_bias_correction": correct, "similarity_settings_hash": config_sha256(similarity)}


def load_experiment_config(config_path, require_enabled=True):
    try:
        import yaml
    except ImportError as exc:
        raise ImportError("PyYAML is required. Install PyYAML on your Databricks environment, then rerun.") from exc

    # Reject duplicate YAML keys rather than silently replacing a weight/setting.
    class UniqueLoader(yaml.SafeLoader):
        pass
    def unique_mapping(loader, node, deep=False):
        result = {}
        for key_node, value_node in node.value:
            key = loader.construct_object(key_node, deep=deep)
            if key in result:
                raise ValueError(f"Duplicate YAML key {key!r} on line {key_node.start_mark.line + 1}.")
            result[key] = loader.construct_object(value_node, deep=deep)
        return result
    UniqueLoader.add_constructor(yaml.resolver.BaseResolver.DEFAULT_MAPPING_TAG, unique_mapping)
    path = Path(config_path).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f"Configuration not found: {path}. Put experiments.yaml in configs beside this notebook, or set CONFIG_PATH to its absolute /Workspace/... path.")
    original_text = path.read_text(encoding="utf-8")
    raw = yaml.load(original_text, Loader=UniqueLoader)
    sections = {"schema_version", "data", "outputs", "tracking", "monitoring", "similarity_defaults", "experiments"}
    # Downstream stages share this file without changing 05's semantic inputs.
    _keys(raw, sections | {"forecast", "evaluation"}, "configuration", sections)
    if isinstance(raw["schema_version"], bool) or raw["schema_version"] != 1:
        raise ValueError("Only schema_version: 1 is supported.")
    _keys(raw["data"], {"revision", "ai_attributes_table", "source_version", "prediction_as_of_date"}, "data",
          {"revision", "ai_attributes_table", "source_version", "prediction_as_of_date"})
    data = copy.deepcopy(raw["data"])
    data["revision"] = _text(data["revision"], "data.revision")
    data["ai_attributes_table"] = _table(data["ai_attributes_table"], "data.ai_attributes_table")
    if data["source_version"] is not None:
        data["source_version"] = _integer(data["source_version"], "data.source_version")
    data["prediction_as_of_date"] = _date(data["prediction_as_of_date"], "data.prediction_as_of_date")
    output_keys = {"like_games_table", "runs_table", "attempts_table"}
    _keys(raw["outputs"], output_keys, "outputs", output_keys)
    outputs = {k: _table(v, f"outputs.{k}") for k, v in raw["outputs"].items()}
    if len(set(outputs.values())) != len(outputs) or data["ai_attributes_table"] in outputs.values():
        raise ValueError("Output tables must be distinct and cannot be the AI source table.")
    _keys(raw["tracking"], {"experiment_path"}, "tracking", {"experiment_path"})
    tracking_path = _text(raw["tracking"]["experiment_path"], "tracking.experiment_path")
    if not tracking_path.startswith("/"):
        raise ValueError("tracking.experiment_path must start with /.")
    _keys(raw["monitoring"], {"history_rows"}, "monitoring", {"history_rows"})
    history_rows = _integer(raw["monitoring"]["history_rows"], "monitoring.history_rows", 1)
    defaults = validate_similarity_settings(raw["similarity_defaults"])
    if not isinstance(raw["experiments"], list) or not raw["experiments"]:
        raise ValueError("experiments must be a nonempty list.")
    experiments, names = [], set()
    for index, item in enumerate(raw["experiments"]):
        _keys(item, {"name", "enabled", "overrides", "forecast"}, f"experiments[{index}]", {"name"})
        name = _text(item["name"], f"experiments[{index}].name")
        if name in names:
            raise ValueError(f"Duplicate experiment name: {name!r}")
        names.add(name)
        enabled = item.get("enabled", True)
        if not isinstance(enabled, bool):
            raise ValueError(f"{name}.enabled must be true or false.")
        overrides = item.get("overrides", {})
        _keys(overrides, defaults, f"{name}.overrides")
        merged = copy.deepcopy(defaults)
        for key, value in overrides.items():
            if key in {"raw_weights", "neighbor_limits"}:
                _keys(value, defaults[key], f"{name}.overrides.{key}")
                merged[key].update(copy.deepcopy(value))
            else:
                merged[key] = copy.deepcopy(value)
        merged = validate_similarity_settings(merged)
        forecast = named_forecast_settings(item, merged)
        experiments.append({"name": name, "enabled": enabled, "settings": merged, "forecast": forecast})
    if require_enabled and not any(x["enabled"] for x in experiments):
        raise ValueError("Enable at least one experiment in the YAML file.")
    return {"data": data, "outputs": outputs, "tracking": {"experiment_path": tracking_path},
            "monitoring": {"history_rows": history_rows}, "experiments": experiments,
            "config_path": str(path), "config_text": original_text,
            "yaml_sha256": hashlib.sha256(original_text.encode("utf-8")).hexdigest()}


def similarity_work_identity(experiment, data, source_version, selection_history):
    semantic = {"implementation_version": SIMILARITY_IMPLEMENTATION_VERSION,
                "data_revision": data["revision"], "source_table": data["ai_attributes_table"],
                "source_table_version": int(source_version),
                "prediction_as_of_date": data["prediction_as_of_date"],
                "prediction_as_of_source": data.get("prediction_as_of_source", "configured"),
                "similarity": copy.deepcopy(experiment["settings"]),
                "selection_history": copy.deepcopy(selection_history)}
    digest = config_sha256(semantic)
    return digest, semantic


def preflight_similarity_tables(spark, config):
    """Refuse to mix pre-ledger legacy generations with the new history."""
    results, runs = (config["outputs"][x] for x in ["like_games_table", "runs_table"])
    if spark.catalog.tableExists(results):
        frame = spark.table(results)
        if frame.limit(1).count():
            if "similarity_work_key" not in frame.columns or not spark.catalog.tableExists(runs):
                raise ValueError("Previous similarity results use the old format. Run 00_Reset_Similarity_Pipeline once before the first new batch. No results were overwritten.")
            ledger = spark.table(runs)
            if not {"work_key", "experiment_id", "metadata_json"}.issubset(ledger.columns):
                raise ValueError("The similarity status table has an incompatible schema. Run the one-time reset before starting.")
            if frame.select("experiment_id").distinct().join(ledger.select("experiment_id").distinct(), "experiment_id", "left_anti").limit(1).count():
                raise ValueError("Similarity rows without a matching status record were found. Resolve/reset them before running a new batch.")


def resolve_similarity_source_version(spark, config):
    """A data revision pins one source snapshot across later manually run batches."""
    from pyspark.sql import functions as F
    data, runs = config["data"], config["outputs"]["runs_table"]
    pinned = []
    if spark.catalog.tableExists(runs):
        ledger = spark.table(runs)
        if "metadata_json" not in ledger.columns:
            raise ValueError("Incompatible similarity status table; perform the one-time reset first.")
        pinned = [int(r["source_table_version"]) for r in ledger.where(
            (F.get_json_object("metadata_json", "$.data_revision") == data["revision"])
            & (F.col("source_path") == data["ai_attributes_table"])
        ).select("source_table_version").distinct().collect()]
    if len(set(pinned)) > 1:
        raise ValueError("This data revision has multiple source snapshots. Choose a new revision after resolving the history.")
    if pinned:
        version = pinned[0]
        if data["source_version"] is not None and data["source_version"] != version:
            raise ValueError(f"Revision {data['revision']!r} is already pinned to Delta version {version}. Change data.revision to use another version.")
        return version
    if data["source_version"] is not None:
        return data["source_version"]
    return int(spark.sql(f"DESCRIBE HISTORY {quoted_table(data['ai_attributes_table'])}").agg(F.max("version").alias("version")).first()["version"])


FORECAST_SOURCE_KEYS = {"sales", "lease", "performance"}
FORECAST_OUTPUT_KEYS = {"demand", "performance", "neighbors", "runs", "attempts"}
DEFAULT_HISTORY_MONTHS = {"sales_min_observed_months": 6, "rental_min_observed_months": 9}
FORECAST_SETTING_KEYS = {"distance_cutoff", "minimum_neighbors", "horizon_months", "slot_counts",
                         "lease_value_column", "lease_metric_name", "lease_curve_label", "performance_metrics"}


def load_forecast_config(config_path):
    # Both stages use the same source definitions and selection options.
    common = load_experiment_config(config_path, require_enabled=False)
    import yaml
    raw = yaml.safe_load(common["config_text"])
    forecast = raw.get("forecast")
    forecast_keys = {"data_revision", "sources", "source_versions", "settings", "outputs", "tracking", "selection", "monitoring"}
    if isinstance(forecast, dict) and "experiments" in forecast:
        raise ValueError("Move forecast.experiments into the named entries in the top-level experiments list.")
    _keys(forecast, forecast_keys, "forecast", forecast_keys)
    revision = _text(forecast["data_revision"], "forecast.data_revision")
    _keys(forecast["sources"], FORECAST_SOURCE_KEYS, "forecast.sources", FORECAST_SOURCE_KEYS)
    sources = {k: _table(v, f"forecast.sources.{k}") for k, v in forecast["sources"].items()}
    _keys(forecast["source_versions"], FORECAST_SOURCE_KEYS, "forecast.source_versions", FORECAST_SOURCE_KEYS)
    versions = {k: None if v is None else _integer(v, f"forecast.source_versions.{k}")
                for k, v in forecast["source_versions"].items()}
    _keys(forecast["outputs"], FORECAST_OUTPUT_KEYS, "forecast.outputs", FORECAST_OUTPUT_KEYS)
    outputs = {k: _table(v, f"forecast.outputs.{k}") for k, v in forecast["outputs"].items()}
    protected_inputs = {common["data"]["ai_attributes_table"], *sources.values(), *common["outputs"].values()}
    if len(set(outputs.values())) != len(outputs) or set(outputs.values()) & protected_inputs:
        raise ValueError("Forecast output tables must be distinct and cannot replace any source or similarity tables.")
    _keys(forecast["settings"], FORECAST_SETTING_KEYS | set(DEFAULT_HISTORY_MONTHS), "forecast.settings", FORECAST_SETTING_KEYS)
    settings = copy.deepcopy(forecast["settings"])
    for name, default in DEFAULT_HISTORY_MONTHS.items():
        settings[name] = _integer(settings.get(name, default), f"forecast.settings.{name}", 1)
    cutoff = settings["distance_cutoff"]
    if isinstance(cutoff, bool) or not isinstance(cutoff, Real) or not math.isfinite(float(cutoff)) or cutoff < 0:
        raise ValueError("forecast.settings.distance_cutoff must be finite and >= 0.")
    settings["distance_cutoff"] = float(cutoff)
    settings["minimum_neighbors"] = _integer(settings["minimum_neighbors"], "minimum_neighbors")
    settings["horizon_months"] = _integer(settings["horizon_months"], "horizon_months", 1)
    slots = settings["slot_counts"]
    if not isinstance(slots, list) or any(isinstance(x, bool) or not isinstance(x, Real) or not math.isfinite(float(x)) or x <= 0 for x in slots):
        raise ValueError("slot_counts must be a list of finite positive numbers; [] means all positive counts.")
    if len(set(slots)) != len(slots):
        raise ValueError("slot_counts cannot contain duplicates.")
    settings["slot_counts"] = sorted(float(x) for x in slots)
    for key in ["lease_value_column", "lease_metric_name", "lease_curve_label"]:
        settings[key] = _text(settings[key], key)
    if not isinstance(settings["performance_metrics"], list) or not settings["performance_metrics"]:
        raise ValueError("performance_metrics must be a nonempty list.")
    settings["performance_metrics"] = [_text(x, "performance_metrics entry") for x in settings["performance_metrics"]]
    if len(set(settings["performance_metrics"])) != len(settings["performance_metrics"]):
        raise ValueError("performance_metrics cannot contain duplicates.")
    settings["performance_metrics"] = sorted(settings["performance_metrics"])
    _keys(forecast["tracking"], {"experiment_path"}, "forecast.tracking", {"experiment_path"})
    mlflow_path = _text(forecast["tracking"]["experiment_path"], "forecast.tracking.experiment_path")
    if not mlflow_path.startswith("/"):
        raise ValueError("forecast.tracking.experiment_path must be an absolute workspace path.")
    _keys(forecast["selection"], {"similarity_experiment_ids"}, "forecast.selection", {"similarity_experiment_ids"})
    ids = forecast["selection"]["similarity_experiment_ids"]
    if not isinstance(ids, list):
        raise ValueError("similarity_experiment_ids must be a list; [] discovers all completed 05 experiments.")
    ids = [_text(x, "similarity_experiment_ids entry") for x in ids]
    if len(set(ids)) != len(ids):
        raise ValueError("similarity_experiment_ids cannot contain duplicates.")
    _keys(forecast["monitoring"], {"history_rows"}, "forecast.monitoring", {"history_rows"})
    return {"common": common,
            "data_revision": revision, "sources": sources, "source_versions": versions,
            "settings": settings, "outputs": outputs, "tracking": {"experiment_path": mlflow_path},
            "selection": {"similarity_experiment_ids": ids},
            "monitoring": {"history_rows": _integer(forecast["monitoring"]["history_rows"], "forecast.monitoring.history_rows", 1)}}


def neighbor_selection_policy(config, versions):
    keys = ("sales_min_observed_months", "rental_min_observed_months", "distance_cutoff",
            "minimum_neighbors", "lease_value_column", "performance_metrics", "slot_counts")
    return {
        "data_revision": config["data_revision"],
        "source_paths": dict(config["sources"]),
        "source_versions": dict(versions),
        "settings": {key: config["settings"][key] for key in keys},
    }


def resolve_forecast_source_versions(spark, config):
    from pyspark.sql import functions as F
    table = config["outputs"]["runs"]
    matches = []
    if spark.catalog.tableExists(table):
        ledger = spark.table(table)
        if "metadata_json" not in ledger.columns:
            raise ValueError("Legacy forecast manifest found. Reset scope '06_07' before using new 06; preserve 05.")
        matches = ledger.where(F.get_json_object("metadata_json", "$.data_revision") == config["data_revision"]).select(
            "source_versions_json", "config_json").distinct().collect()
    versions = None
    for row in matches:
        previous_config = json.loads(row["config_json"])
        if previous_config["source_paths"] != config["sources"]:
            raise ValueError("This forecast data revision is already assigned to other source tables. Choose another forecast.data_revision.")
        candidate = {k: int(v) for k, v in json.loads(row["source_versions_json"]).items()}
        if versions is not None and candidate != versions:
            raise ValueError("This forecast revision contains conflicting source snapshots; resolve its history before running.")
        versions = candidate
    # A completed or interrupted 05 run can be the first writer for this revision.
    similarity_runs = config["common"]["outputs"]["runs_table"]
    if spark.catalog.tableExists(similarity_runs):
        selections = (spark.table(similarity_runs)
                      .where(F.get_json_object("config_json", "$.selection_history.data_revision") == config["data_revision"])
                      .select(F.get_json_object("config_json", "$.selection_history").alias("history")).distinct())
        for row in selections.collect():
            history = json.loads(row["history"])
            if history["source_paths"] != config["sources"]:
                raise ValueError("This data revision is pinned to different selection sources.")
            candidate = history["source_versions"]
            if versions is not None and versions != candidate:
                raise ValueError("Neighbor selection and forecast ledgers disagree on this revision's snapshots.")
            versions = candidate
    if versions is not None:
        for key, requested in config["source_versions"].items():
            if requested is not None and requested != versions[key]:
                raise ValueError(f"Revision is pinned to {key} version {versions[key]}; change forecast.data_revision to refresh.")
        return versions
    return {key: int(requested) if requested is not None else int(
        spark.sql(f"DESCRIBE HISTORY {quoted_table(config['sources'][key])}").agg(F.max("version").alias("v")).first()["v"])
        for key, requested in config["source_versions"].items()}


In [0]:
"""Notebook 06 YAML validation, immutable work keys, and source-revision pinning."""
import copy
import hashlib
import json
import math
from numbers import Integral, Real

FORECAST_IMPLEMENTATION_VERSION = "06_named_ranked_demand_v1"
def forecast_work_identity(upstream, config, source_versions, experiment):
    # The named forecast and its weights identify an independently saved run.
    settings = {k: copy.deepcopy(v) for k, v in config["settings"].items() if k != "lease_curve_label"}
    semantic = {
        "implementation_version": FORECAST_IMPLEMENTATION_VERSION,
        "forecast_data_revision": config["data_revision"],
        "similarity_experiment_id": upstream["experiment_id"],
        "similarity_work_key": upstream["work_key"],
        "similarity_config_hash": upstream["config_hash"],
        "similarity_result_hash": upstream["result_hash"],
        "source_paths": copy.deepcopy(config["sources"]),
        "source_versions_resolved": {k: int(source_versions[k]) for k in sorted(FORECAST_SOURCE_KEYS)},
        **settings,
        "prediction_date_rule": "saved similarity prediction_date, falling back to target game_release_date",
        "neighbor_observation_rule": "source_month < first_day_of_target_prediction_month",
        "reporting_assumption": "completed prior month available; publication timestamps not supplied",
        "calendar_rule": "target forecast starts at explicit prediction month; neighbor age starts at first historically observed source month",
        "target_actual_policy": "predictions only; actuals loaded independently by versioned evaluation",
        "sales_fill": "internal missing monthly rows zero, then cumulative sum; no trailing fill",
        "lease_fill": "internal missing monthly levels forward-filled; no trailing fill",
        "performance_fill": "internal missing per-metric neighbor statistics forward-filled; leading null remains; no trailing fill",
        "performance_weighting": "positive no_of_slots; per-metric nonnull denominator",
        "neighbor_selection": "consume final saved 05 channel lists in stored rank order",
        "commercial_eligibility": "enforced in 05 before ranking on the pinned source snapshots",
        "demand_curve_fit": "OLS without intercept: x, (x,x^2), log1p(x). Use 2/3/2 supported months; full horizon; shift to nonnegative; centered R2 on supported months",
        "p3_parent_after_cutoff": "applied in 05 before publication",
        "bands": "population standard deviation across contributing neighbors; mean +/- 2 std, not calibrated intervals",
    }
    semantic["named_experiment"] = copy.deepcopy(experiment)
    upstream_config = json.loads(upstream["config_json"])
    semantic["similarity_data"] = {k: upstream_config.get(k) for k in (
        "data_revision", "source_table", "source_table_version", "prediction_as_of_source", "prediction_as_of_date")}
    return config_sha256(semantic), semantic


def preflight_forecast_tables(spark, config):
    outputs = config["outputs"]
    for label in ["demand", "performance", "neighbors"]:
        path = outputs[label]
        if spark.catalog.tableExists(path):
            frame = spark.table(path)
            if frame.limit(1).count():
                if "forecast_work_key" not in frame.columns or not spark.catalog.tableExists(outputs["runs"]):
                    raise ValueError("Legacy forecast results were found. Clear only the old 06/07 outputs using reset scope '06_07' before this migration. Preserve your new 05 results.")
                runs = spark.table(outputs["runs"])
                if not {"work_key", "forecast_run_id", "metadata_json"}.issubset(runs.columns):
                    raise ValueError("The forecast status table uses the old schema. Migrate/reset scope '06_07'; preserve 05.")
                if frame.select("forecast_run_id").distinct().join(runs.select("forecast_run_id").distinct(), "forecast_run_id", "left_anti").limit(1).count():
                    raise ValueError("Forecast rows without status records were found. Resolve them before retrying this batch.")




## Shared source preparation and forecasting

Reusable computation functions. Actuals are excluded from target prediction production.


In [0]:
"""Pure computation for notebook 06; persistence and orchestration live elsewhere.

Predictions depend on historical neighbor observations only. Target actuals are
deliberately absent and are joined by evaluation against an independent snapshot.
"""
import copy
import math
from numbers import Integral, Real


def validate_forecast_settings(settings):
    values = {
        "distance_cutoff": settings.get("distance_cutoff", 0.4),
        "minimum_neighbors": settings.get("minimum_neighbors", 3),
        "horizon_months": settings.get("horizon_months", 12),
        "slot_counts": list(settings.get("slot_counts", [])),
        "lease_value_column": settings.get("lease_value_column", "source_row_count"),
        "lease_metric_name": settings.get("lease_metric_name", "monthly_rental_records"),
        "lease_curve_label": settings.get("lease_curve_label", "Monthly rental records"),
        "performance_metrics": list(settings.get("performance_metrics", [
            "avg_coin_in_index_vs_house", "avg_theo_net_win_index_vs_house"])),
    }
    for name, default in DEFAULT_HISTORY_MONTHS.items():
        values[name] = _integer(settings.get(name, default), name, 1)
    cutoff = values["distance_cutoff"]
    if isinstance(cutoff, bool) or not isinstance(cutoff, Real) or not math.isfinite(float(cutoff)) or cutoff < 0:
        raise ValueError("distance_cutoff must be a finite nonnegative number.")
    for name, minimum in [("minimum_neighbors", 0), ("horizon_months", 1)]:
        value = values[name]
        if isinstance(value, bool) or not isinstance(value, Integral) or value < minimum:
            raise ValueError(f"{name} must be an integer >= {minimum}.")
    slots = values["slot_counts"]
    if any(isinstance(value, bool) or not isinstance(value, Real) or not math.isfinite(float(value)) or value <= 0 for value in slots):
        raise ValueError("slot_counts must contain positive finite numbers.")
    if len(slots) != len(set(slots)):
        raise ValueError("slot_counts cannot contain duplicates.")
    metrics = values["performance_metrics"]
    if not metrics or any(not isinstance(metric, str) or not metric for metric in metrics) or len(metrics) != len(set(metrics)):
        raise ValueError("performance_metrics must contain distinct nonempty column names.")
    for name in ["lease_value_column", "lease_metric_name", "lease_curve_label"]:
        if not isinstance(values[name], str) or not values[name].strip():
            raise ValueError(f"{name} must be a nonempty string.")
    values["distance_cutoff"] = float(cutoff)
    values["minimum_neighbors"] = int(values["minimum_neighbors"])
    values["horizon_months"] = int(values["horizon_months"])
    values["slot_counts"] = sorted(float(value) for value in slots)
    return values


def commercial_curve_reference(records, revenue_type, prediction_date, horizon=12):
    """Pure monthly reference for historical cutoff, bounded fills, and sales sums.

    records use {month, value}. Result rows use month_index, month, value,
    is_imputed. This reference is small-data validation, never a source collector.
    """
    import pandas as pd
    if revenue_type not in {"sales", "lease"}:
        raise ValueError("revenue_type must be sales or lease.")
    if prediction_date is None or pd.isna(pd.Timestamp(prediction_date)):
        return []
    cutoff = pd.Timestamp(prediction_date).to_period("M").to_timestamp()
    observations = {}
    for record in records:
        month = pd.to_datetime(record.get("month"), errors="coerce")
        value = pd.to_numeric(record.get("value"), errors="coerce")
        if pd.isna(month) or pd.isna(value) or not math.isfinite(float(value)):
            continue
        month = month.to_period("M").to_timestamp()
        if month < cutoff:
            observations[month] = observations.get(month, 0.0) + float(value)
    if not observations:
        return []
    start, last = min(observations), max(observations)
    months = pd.date_range(start, last, freq="MS")[:horizon]
    rows, running, previous = [], 0.0, None
    for index, month in enumerate(months, 1):
        observed = observations.get(month)
        if observed is not None:
            previous = observed
        if revenue_type == "sales":
            running += 0.0 if observed is None else observed
            value = running
        else:
            value = previous
        rows.append({"month_index": index, "month": month.date(), "value": value, "is_imputed": observed is None})
    return rows


def weighted_index_reference(records, metrics):
    """Pure reference of duplicate-slot averaging and nonnull metric weighting.

    records represent one relative month and use neighbor_game/no_of_slots plus
    metric names. Duplicate rows in a neighbor/slot bucket are averaged first.
    """
    buckets = {}
    for record in records:
        try:
            slots = float(record.get("no_of_slots"))
        except (ValueError, TypeError):
            continue
        if not math.isfinite(slots) or slots <= 0:
            continue
        key = (record.get("neighbor_game", "neighbor"), slots)
        bucket = buckets.setdefault(key, {metric: [] for metric in metrics})
        for metric in metrics:
            try:
                value = float(record.get(metric))
            except (ValueError, TypeError):
                continue
            if math.isfinite(value):
                bucket[metric].append(value)
    output = {}
    for metric in metrics:
        weighted = [(slots, sum(bucket[metric]) / len(bucket[metric]))
                    for (_, slots), bucket in buckets.items() if bucket[metric]]
        denominator = sum(slots for slots, _ in weighted)
        mean = sum(slots * value for slots, value in weighted) / denominator if denominator else None
        variance = max(0.0, sum(slots * value * value for slots, value in weighted) / denominator - mean * mean) if denominator else None
        output[metric] = {"mean": mean, "std": None if variance is None else math.sqrt(variance), "weight_sum": denominator}
    return output


def _functions():
    from pyspark.sql import functions as F
    return F


def _normalized(column):
    F = _functions()
    return F.regexp_replace(F.lower(F.trim(column.cast("string"))), r"\s+", " ")


def _valid_name(column):
    value = _normalized(column)
    return value.isNotNull() & ~value.isin("", "none", "null", "nan", "na", "n/a", "<na>")


def _finite(column):
    F = _functions()
    value = column.cast("double")
    return F.when(value.isNotNull() & ~F.isnan(value) & (F.abs(value) != float("inf")), value)


def _require_columns(frame, columns, label):
    missing = sorted(set(columns) - set(frame.columns))
    if missing:
        raise ValueError(f"{label} is missing columns: {missing}")


def monthly_source_frames(source_frames, values):
    """Build one row per commercial game-month and per Eilers game-month/slot bucket."""
    F = _functions()
    missing = {"sales", "lease", "performance"} - set(source_frames)
    if missing:
        raise ValueError(f"Missing source frames: {sorted(missing)}")
    sales, lease, perf = (source_frames[name] for name in ["sales", "lease", "performance"])
    _require_columns(sales, ["normalized_game_name", "revenue_type", "beginning_month_date", "expr1_sum"], "Sales")
    _require_columns(lease, ["theme_name_eilers", "month", values["lease_value_column"]], "Rental")
    _require_columns(perf, ["game_name", "own_status", "yearmonth", "no_of_slots", *values["performance_metrics"]], "Eilers")
    sales_monthly = (
        sales.where(_normalized(F.col("revenue_type")).isin("sale", "sales"))
        .where(_valid_name(F.col("normalized_game_name")))
        .select(_normalized(F.col("normalized_game_name")).alias("_source_key"), F.lit("sales").alias("revenue_type"),
                F.trunc(F.to_date("beginning_month_date"), "month").alias("_calendar_month"),
                _finite(F.col("expr1_sum")).alias("_monthly_value"))
    )
    lease_monthly = (
        lease.where(_valid_name(F.col("theme_name_eilers")))
        .select(_normalized(F.col("theme_name_eilers")).alias("_source_key"), F.lit("lease").alias("revenue_type"),
                F.trunc(F.to_date("month"), "month").alias("_calendar_month"),
                _finite(F.col(values["lease_value_column"])).alias("_monthly_value"))
    )
    commercial = (
        sales_monthly.unionByName(lease_monthly)
        .where(F.col("_calendar_month").isNotNull() & F.col("_monthly_value").isNotNull())
        .groupBy("revenue_type", "_source_key", "_calendar_month")
        .agg(F.sum("_monthly_value").alias("_monthly_value"))
    )
    performance = (
        perf.select(_normalized(F.col("game_name")).alias("_game_key"),
                    _normalized(F.col("own_status")).alias("own_status"),
                    F.trunc(F.to_date(F.col("yearmonth").cast("string"), "yyyy-MM-dd"), "month").alias("_perf_month"),
                    _finite(F.col("no_of_slots")).alias("no_of_slots"),
                    *[_finite(F.col(metric)).alias(metric) for metric in values["performance_metrics"]])
        .where(_valid_name(F.col("_game_key")) & F.col("own_status").isin("owned", "leased")
               & F.col("_perf_month").isNotNull() & (F.col("no_of_slots") > 0))
    )
    if values["slot_counts"]:
        performance = performance.where(F.col("no_of_slots").isin(values["slot_counts"]))
    # Original 06 averages duplicates per game/month/slot bucket for neighbors.
    # This aggregation is independent of experiment/target and can be shared.
    performance_monthly = performance.groupBy("_game_key", "own_status", "_perf_month", "no_of_slots").agg(
        *[F.avg(metric).alias(metric) for metric in values["performance_metrics"]])
    return commercial, performance_monthly


def prepare_forecast_sources(spark, source_frames, settings):
    """Normalize/aggregate pinned source frames once for a whole 06 batch.

    The two materialized caches contain monthly commercial records and monthly
    Eilers slot buckets. They carry no target observations or target calendars.
    Call release_forecast_sources() in the batch's finally block.
    """
    F = _functions()
    values = validate_forecast_settings(settings)
    commercial, performance_monthly = monthly_source_frames(source_frames, values)
    cached = []
    try:
        commercial = commercial.cache()
        cached.append(commercial)
        performance_monthly = performance_monthly.cache()
        cached.append(performance_monthly)
        diagnostics = {"commercial_monthly_rows": commercial.count(), "performance_monthly_slot_rows": performance_monthly.count()}
        return {"commercial_monthly": commercial, "performance_neighbor_monthly": performance_monthly,
                "cached_frames": cached, "diagnostics": diagnostics, "settings": values}
    except BaseException:
        for frame in cached:
            frame.unpersist()
        raise


def release_forecast_sources(prepared):
    for frame in prepared.get("cached_frames", []):
        frame.unpersist()


def release_forecast_outputs(result):
    for frame in result.get("frames", {}).values():
        frame.unpersist()


def _commercial_curves(monthly, keys, horizon):
    """Align neighbor age months; fill only between as-of source observations."""
    F = _functions()
    from pyspark.sql.window import Window
    bounds = monthly.groupBy(*keys).agg(
        F.min("_calendar_month").alias("curve_start_month"),
        F.max("_calendar_month").alias("last_observed_source_month"),
        F.count("_monthly_value").alias("source_months_observed"))
    scaffold = (
        bounds.withColumn("_last_month", F.least(
            F.floor(F.months_between("last_observed_source_month", "curve_start_month")).cast("int") + 1, F.lit(horizon)))
        .withColumn("month_index", F.explode(F.sequence(F.lit(1), F.col("_last_month"))))
        .withColumn("_calendar_month", F.add_months("curve_start_month", F.col("month_index") - 1)).drop("_last_month")
    )
    window = Window.partitionBy(*keys).orderBy("month_index").rowsBetween(Window.unboundedPreceding, Window.currentRow)
    return (
        scaffold.join(monthly.select(*keys, "_calendar_month", "_monthly_value"), keys + ["_calendar_month"], "left")
        .withColumn("actual_is_imputed", F.col("_monthly_value").isNull())
        .withColumn("actual_is_observed", F.col("_monthly_value").isNotNull())
        .withColumn("_filled_monthly_value", F.when(F.col("revenue_type") == "sales", F.coalesce("_monthly_value", F.lit(0.0)))
                    .otherwise(F.last("_monthly_value", ignorenulls=True).over(window)))
        .withColumn("source_curve_value", F.when(F.col("revenue_type") == "sales", F.sum("_filled_monthly_value").over(window))
                    .otherwise(F.col("_filled_monthly_value")))
        .withColumn("actual_contains_imputed_history", F.when(F.col("revenue_type") == "sales",
                    F.max(F.col("actual_is_imputed").cast("int")).over(window) > 0).otherwise(F.col("actual_is_imputed")))
    )


def weight_demand_benchmark(baseline, rank_weights):
    """Compute the moments of each month's ranked, available commercial neighbors."""
    F = _functions()
    # A single weight applies to all ranks. Equal weights use the original aggregate.
    if len(set(rank_weights)) == 1:
        return baseline.drop("_values")
    weights = F.array(*[F.lit(w) for w in rank_weights])
    # Reuse the last tier if more neighbors are available than explicit ranks.
    # Tied values are interchangeable for the weighted mean and dispersion.
    ranked = F.transform(
        F.sort_array("_values", asc=False),
        lambda value, index: F.struct(
            value.alias("value"),
            F.element_at(weights, F.least(index + 1, F.lit(len(rank_weights)))).alias("weight")),
    )
    weighted = baseline.withColumn("_ranked", ranked)
    total = F.aggregate("_ranked", F.lit(0.0), lambda acc, p: acc + p["weight"])
    weighted_sum = F.aggregate("_ranked", F.lit(0.0), lambda acc, p: acc + p["value"] * p["weight"])
    weighted = (weighted.withColumn("_total", total)
                .withColumn("forecast_value", F.when(F.col("_total") > 0, weighted_sum / F.col("_total"))))
    deviations = F.aggregate("_ranked", F.lit(0.0), lambda acc, p:
                             acc + p["weight"] * F.pow(p["value"] - F.col("forecast_value"), 2))
    return (weighted.withColumn("_stddev", F.when(F.col("_total") > 0, F.sqrt(deviations / F.col("_total"))))
            .drop("_values", "_ranked", "_total"))


def fit_demand_curves(points, horizon):
    """Fit one target/channel benchmark on its actual month indices."""
    import numpy as np
    observations = sorted((int(p[0]), float(p[1])) for p in points
                          if p[1] is not None and math.isfinite(p[1]))
    count = len(observations)
    result = {"linear": None, "poly2": None, "log": None, "fit_months": count,
              "last_fit_month": observations[-1][0] if count else None,
              "best_model": None, "best_r2": None}
    if count < 2:
        return result
    indices, y = zip(*observations)
    x = np.asarray(indices, dtype=float)
    y = np.asarray(y, dtype=float)
    months = np.arange(1, horizon + 1, dtype=float)
    support = np.asarray(indices, dtype=int) - 1
    total_ss = float(np.square(y - y.mean()).sum())
    designs = [
        ("linear", x[:, None], months[:, None]),
        ("log", np.log1p(x)[:, None], np.log1p(months)[:, None]),
        ("poly2", np.column_stack((x, x*x)), np.column_stack((months, months*months))),
    ]
    for name, design, grid in designs:
        # Require one more observed month than coefficients for each model.
        if count <= design.shape[1]:
            continue
        coefficients, _, rank, _ = np.linalg.lstsq(design, y, rcond=None)
        if rank != design.shape[1]:
            continue
        curve = grid @ coefficients
        curve -= min(0.0, float(curve.min()))
        if not np.all(np.isfinite(curve)):
            continue
        result[name] = curve.tolist()
        if total_ss > 0:
            r2 = 1.0 - float(np.square(y - curve[support]).sum()) / total_ss
            # Prefer the earlier, simpler form on a numerical tie.
            if math.isfinite(r2) and (result["best_r2"] is None or r2 > result["best_r2"] + 1e-12):
                result["best_model"], result["best_r2"] = name, r2
    return result


def generate_forecast(spark, like_games_df, prepared, context, settings, rank_weights):
    """Generate both channels and Eilers populations for every canonical target.

    Returned frames are cached for coverage collection and downstream publication;
    callers must release them via release_forecast_outputs() in a finally block.
    Only retained-neighbor keys join source tables. Target mappings are metadata
    used for self-exclusion, never gates for inference or target source lookups.
    """
    F = _functions()
    from pyspark.sql.window import Window
    values = validate_forecast_settings(settings)
    # Presentation is frozen separately by the durable run record. It does not
    # affect shared source preparation, so a retry may use its original label.
    shared_comparison = {key: value for key, value in prepared["settings"].items() if key != "lease_curve_label"}
    requested_comparison = {key: value for key, value in values.items() if key != "lease_curve_label"}
    if requested_comparison != shared_comparison:
        raise ValueError("Shared source preprocessing belongs to different forecast settings.")
    required_context = {"forecast_run_id", "experiment_id"}
    if not all(context.get(name) for name in required_context):
        raise ValueError("context requires experiment_id and forecast_run_id.")
    _require_columns(like_games_df, ["join_game_name", "game_release_date", "sales_similar_games", "rental_similar_games", "eilers_similar_games", "neighbor_history_threshold_months"], "History-qualified 05 generation")
    horizon, metrics = values["horizon_months"], values["performance_metrics"]
    source = like_games_df
    optional_strings = ["parent_game", "parent_game_match_status", "resolved_parent_game_name", "game_category",
                        "parent_own_status", "game_classification", "own_status", "parent_game_process", "distance_method",
                        "to_sales_name_match", "to_rental_name_match", "to_eilers_ts_name_match", "forecast_eligibility_status"]
    for name in optional_strings:
        if name not in source.columns:
            source = source.withColumn(name, F.lit(None).cast("string"))
    for name in ["has_sales_match", "has_rental_match", "has_eilers_ts_match", "parent_in_sales_neighbors", "parent_in_rental_neighbors"]:
        if name not in source.columns:
            source = source.withColumn(name, F.lit(False))
        else:
            source = source.withColumn(name, F.coalesce(_normalized(F.col(name)).isin("true", "1", "yes", "y"), F.lit(False)))
    if "feature_group_weights" not in source.columns:
        source = source.withColumn("feature_group_weights", F.from_json(F.lit("{}"), "map<string,double>"))
    if "prediction_date" not in source.columns:
        source = source.withColumn("prediction_date", F.to_date("game_release_date"))
    else:
        source = source.withColumn("prediction_date", F.coalesce(F.to_date("prediction_date"), F.to_date("game_release_date")))
    if "prediction_date_source" not in source.columns:
        source = source.withColumn("prediction_date_source", F.when(F.col("prediction_date").isNotNull(), F.lit("game_release_date")).otherwise(F.lit("missing")))
    lineage = {
        "experiment_id": context["experiment_id"], "experiment_name": context.get("experiment_name", context.get("experiment_readable_name", context["experiment_id"])),
        "forecast_run_id": context["forecast_run_id"], "forecast_config_hash": context.get("config_hash", context.get("forecast_config_hash")),
        "similarity_mlflow_run_id": context.get("similarity_mlflow_run_id"), "similarity_config_hash": context.get("similarity_config_hash"),
        "similarity_result_hash": context.get("similarity_result_hash"),
    }
    for name, value in lineage.items():
        if value is not None or name not in source.columns:
            source = source.withColumn(name, F.lit(value).cast("string"))
    targets = (
        source.where(_valid_name(F.col("join_game_name")))
        .withColumn("target_game_key", _normalized(F.col("join_game_name")))
        .withColumn("_target_id", F.xxhash64("target_game_key"))
        .withColumn("target_game", F.col("join_game_name"))
        .withColumn("game_release_date", F.to_date("game_release_date"))
        .withColumnRenamed("forecast_eligibility_status", "similarity_forecast_eligibility_status")
    )
    if targets.groupBy("target_game_key").count().where(F.col("count") > 1).limit(1).count():
        raise ValueError("Similarity generation must have one row per normalized target key.")
    metadata = ["_target_id", "target_game", "target_game_key", "game_release_date", "prediction_date", "prediction_date_source",
                "experiment_id", "experiment_name", "distance_method", "feature_group_weights", "similarity_mlflow_run_id",
                "similarity_config_hash", "similarity_result_hash", "forecast_run_id", "forecast_config_hash", "has_sales_match", "has_rental_match", "has_eilers_ts_match",
                "to_sales_name_match", "to_rental_name_match", "to_eilers_ts_name_match", "parent_game", "parent_game_process",
                "parent_game_match_status", "resolved_parent_game_name", "parent_in_sales_neighbors", "parent_in_rental_neighbors",
                "game_category", "parent_own_status", "game_classification", "similarity_forecast_eligibility_status"]
    target_info = targets.select(*metadata, "own_status")
    channels = (
        targets.select(*metadata, F.col("own_status").alias("target_own_status"))
        .withColumn("revenue_type", F.explode(F.array(F.lit("sales"), F.lit("lease"))))
        .withColumn("game_id", F.col("_target_id"))
        .withColumn("own_status", F.when(F.col("revenue_type") == "sales", F.lit("owned")).otherwise(F.lit("leased")))
        .withColumn("metric_name", F.when(F.col("revenue_type") == "sales", F.lit("cumulative_sales")).otherwise(F.lit(values["lease_metric_name"])))
        .withColumn("curve_label", F.when(F.col("revenue_type") == "sales", F.lit("Cumulative sales")).otherwise(F.lit(values["lease_curve_label"])))
        .withColumn("curve_kind", F.when(F.col("revenue_type") == "sales", F.lit("cumulative")).otherwise(F.lit("monthly")))
        .withColumn("_source_key", _normalized(F.when(F.col("revenue_type") == "sales", F.col("to_sales_name_match")).otherwise(F.col("to_rental_name_match"))))
    )

    def saved_neighbors(list_column, source_name):
        # Use exactly the order and membership published by 05.
        expanded = (targets.select("_target_id", "neighbor_history_threshold_months",
                                   F.posexplode(list_column).alias("_position", "_neighbor"))
                    .select("_target_id", "neighbor_history_threshold_months",
                            (F.col("_position") + 1).alias("neighbor_rank"),
                            F.col("_neighbor.join_game_name_key").alias("neighbor_game"),
                            F.col("_neighbor.matched_source_name").alias("matched_source_name"),
                            F.col("_neighbor.distance").alias("distance"),
                            F.col("_neighbor.similarity_score").alias("similarity_score")))
        if source_name == "eilers":
            required = F.lit(None).cast("int")
            threshold = F.lit(None).cast("date")
        else:
            required = F.lit(values[source_name + "_min_observed_months"])
            threshold = F.element_at(F.col("neighbor_history_threshold_months")[source_name], F.col("neighbor_game"))
        return (expanded.withColumn("history_threshold_month", threshold)
                .withColumn("minimum_observed_months_required", required)
                .drop("neighbor_history_threshold_months")
                .withColumn("_neighbor_source_key", _normalized(F.col("matched_source_name")))
                .withColumn("neighbor_type", F.lit(source_name))
                .withColumn("retained_within_cutoff", F.col("distance") <= values["distance_cutoff"])
                .withColumn("retained_as_min_neighbor_fallback", F.col("distance") > values["distance_cutoff"])
                .join(target_info, "_target_id", "inner"))

    sales_neighbors = saved_neighbors("sales_similar_games", "sales")
    rental_neighbors = saved_neighbors("rental_similar_games", "rental")
    eilers_neighbors = saved_neighbors("eilers_similar_games", "eilers")
    all_neighbors = sales_neighbors.unionByName(rental_neighbors).unionByName(eilers_neighbors)
    neighbor_columns = ["_target_id", "neighbor_game", "matched_source_name", "distance"]
    commercial_neighbors = (
        sales_neighbors.select(*neighbor_columns).withColumn("revenue_type", F.lit("sales"))
        .unionByName(rental_neighbors.select(*neighbor_columns).withColumn("revenue_type", F.lit("lease")))
        .withColumn("_source_key", _normalized(F.col("matched_source_name")))
        .join(channels.select("_target_id", "revenue_type", "target_game", "target_game_key", "prediction_date", "own_status", "forecast_run_id"), ["_target_id", "revenue_type"], "inner")
    )
    # Filter source months BEFORE discovering coverage bounds or filling gaps.
    commercial_asof = (
        commercial_neighbors.join(prepared["commercial_monthly"], ["revenue_type", "_source_key"], "inner")
        .where(F.col("_calendar_month") < F.trunc("prediction_date", "month"))
    )
    commercial_keys = ["_target_id", "revenue_type", "_source_key"]
    neighbor_curves = (
        _commercial_curves(commercial_asof.select(*commercial_keys, "_calendar_month", "_monthly_value"), commercial_keys, horizon)
        .join(commercial_neighbors, commercial_keys, "inner")
        .withColumn("neighbor_type", F.when(F.col("revenue_type") == "sales", F.lit("sales")).otherwise(F.lit("rental")))
    )
    baseline = (
        neighbor_curves.groupBy("_target_id", "revenue_type", "month_index")
        .agg(F.avg("source_curve_value").alias("forecast_value"), F.stddev_pop("source_curve_value").alias("_stddev"),
             F.collect_list("source_curve_value").alias("_values"),
             F.countDistinct("_source_key").alias("neighbor_count"), F.sum(F.col("actual_is_imputed").cast("long")).alias("filled_neighbor_count"),
             F.sum(F.col("actual_contains_imputed_history").cast("long")).alias("neighbors_with_imputed_history"))
    )
    baseline = (weight_demand_benchmark(baseline, rank_weights)
        .withColumn("_stddev", F.coalesce("_stddev", F.lit(0.0)))
        .withColumn("observed_neighbor_count", F.col("neighbor_count") - F.col("filled_neighbor_count"))
        .withColumn("high_band_2std", F.col("forecast_value") + 2 * F.col("_stddev"))
        .withColumn("low_band_2std", F.greatest(F.col("forecast_value") - 2 * F.col("_stddev"), F.lit(0.0))).drop("_stddev")
    )
    demand = (
        channels.withColumn("curve_start_month", F.trunc("prediction_date", "month"))
        .withColumn("curve_start_is_inferred", F.lit(True))
        .withColumn("target_curve_starts_before_prediction", F.lit(False))
        .withColumn("month_index", F.explode(F.sequence(F.lit(1), F.lit(horizon))))
        .join(baseline, ["_target_id", "revenue_type", "month_index"], "left")
        .withColumn("forecast_month", F.add_months("curve_start_month", F.col("month_index") - 1))
        .fillna({"neighbor_count": 0, "filled_neighbor_count": 0, "observed_neighbor_count": 0, "neighbors_with_imputed_history": 0})
        .withColumn("baseline_mean_sales", F.when(F.col("revenue_type") == "sales", F.col("forecast_value")))
        .withColumn("baseline_mean_lease", F.when(F.col("revenue_type") == "lease", F.col("forecast_value")))
        .drop("_source_key")
    )
    # Compatibility placeholders: target actuals are exclusively evaluation data.
    demand = demand.withColumn("actual_curve_start_month", F.lit(None).cast("date"))
    for name in ["actual_value", "actual_monthly_observed_value", "actual_cumulative_sales", "actual_monthly_lease"]:
        demand = demand.withColumn(name, F.lit(None).cast("double"))
    for name in ["actual_is_imputed", "actual_is_observed", "actual_contains_imputed_history", "actual_sales_month_zero_filled"]:
        demand = demand.withColumn(name, F.lit(False))

    performance_targets = channels.drop("_source_key", "metric_name", "curve_label", "curve_kind")
    performance_neighbors = (
        eilers_neighbors.select("_target_id", "neighbor_game", "matched_source_name", "distance", "similarity_score")
        .withColumn("_neighbor_key", _normalized(F.col("matched_source_name")))
        .join(performance_targets.select("_target_id", "target_game", "target_game_key", "own_status", "revenue_type", "prediction_date", "forecast_run_id", "forecast_config_hash"), "_target_id", "inner")
    )
    n, p = performance_neighbors.alias("n"), prepared["performance_neighbor_monthly"].alias("p")
    neighbor_perf = (
        n.join(p, (F.col("n._neighbor_key") == F.col("p._game_key")) & (F.col("n.own_status") == F.col("p.own_status"))
               & (F.col("p._perf_month") < F.trunc(F.col("n.prediction_date"), "month")), "inner")
        .select("n.*", "p.no_of_slots", "p._perf_month", *[F.col("p." + metric).alias(metric) for metric in metrics])
    )
    start_window = Window.partitionBy("_target_id", "own_status", "_neighbor_key")
    neighbor_perf = (
        neighbor_perf.withColumn("_neighbor_start_month", F.min("_perf_month").over(start_window))
        .withColumn("month_index", F.floor(F.months_between("_perf_month", "_neighbor_start_month")).cast("int") + 1)
    )
    neighbor_game_keys = ["_target_id", "target_game", "own_status", "_neighbor_key"]
    neighbor_monthly = neighbor_perf.where(F.col("month_index").between(1, horizon))
    aggregations = []
    for metric in metrics:
        weight = F.when(F.col(metric).isNotNull(), F.col("no_of_slots"))
        aggregations.extend([F.sum(weight).alias(f"_{metric}_weight_sum"), F.sum(weight * F.col(metric)).alias(f"_{metric}_weighted_sum"),
                             F.sum(weight * F.pow(F.col(metric), 2)).alias(f"_{metric}_weighted_sq_sum")])
    monthly_stats = neighbor_monthly.groupBy(*neighbor_game_keys, "month_index").agg(*aggregations)
    ranges = neighbor_perf.groupBy(*neighbor_game_keys).agg(F.least(F.max("month_index"), F.lit(horizon)).alias("_last_month"))
    stats = ranges.withColumn("month_index", F.explode(F.sequence(F.lit(1), F.col("_last_month")))).drop("_last_month").join(monthly_stats, neighbor_game_keys + ["month_index"], "left")
    fill_window = Window.partitionBy(*neighbor_game_keys).orderBy("month_index").rowsBetween(Window.unboundedPreceding, Window.currentRow)
    final_aggregations = []
    for metric in metrics:
        columns = [f"_{metric}_weight_sum", f"_{metric}_weighted_sum", f"_{metric}_weighted_sq_sum"]
        observed, filled = f"{metric}_is_observed", f"{metric}_is_filled"
        valid_stats = F.when(F.col(columns[0]) > 0, F.struct(*[F.col(name).alias(name) for name in columns]))
        stats = (stats.withColumn(observed, F.coalesce(F.col(columns[0]) > 0, F.lit(False)))
                 .withColumn("_last_metric_stats", F.last(valid_stats, ignorenulls=True).over(fill_window))
                 .withColumn(filled, ~F.col(observed) & F.col("_last_metric_stats").isNotNull()))
        for name in columns:
            stats = stats.withColumn(name, F.col("_last_metric_stats").getField(name))
        stats = stats.drop("_last_metric_stats")
        final_aggregations.extend([*[F.sum(name).alias(name) for name in columns],
                                   F.count(F.when(F.col(columns[0]) > 0, F.lit(1))).alias(f"{metric}_neighbor_count"),
                                   F.sum(F.col(observed).cast("long")).alias(f"{metric}_observed_neighbor_count"),
                                   F.sum(F.col(filled).cast("long")).alias(f"{metric}_filled_neighbor_count")])
    group_columns = ["_target_id", "target_game", "own_status", "month_index"]
    performance_monthly = stats.groupBy(*group_columns).agg(*final_aggregations)
    for metric in metrics:
        weight, mean, std = F.col(f"_{metric}_weight_sum"), f"{metric}_mean", f"{metric}_std"
        performance_monthly = (
            performance_monthly.withColumn(mean, F.when(weight > 0, F.col(f"_{metric}_weighted_sum") / weight))
            .withColumn(std, F.when(weight > 0, F.sqrt(F.greatest(F.col(f"_{metric}_weighted_sq_sum") / weight - F.pow(F.col(mean), 2), F.lit(0.0)))))
            .withColumn(f"{metric}_lower_2std", F.col(mean) - 2 * F.col(std))
            .withColumn(f"{metric}_higher_2std", F.col(mean) + 2 * F.col(std))
        )
    performance = (
        performance_targets.withColumn("curve_start_month", F.trunc("prediction_date", "month"))
        .withColumn("target_first_observed_month", F.lit(None).cast("date"))
        .withColumn("target_start_is_observed", F.lit(False))
        .withColumn("target_curve_starts_before_prediction", F.lit(False))
        .withColumn("month_index", F.explode(F.sequence(F.lit(1), F.lit(horizon))))
        .withColumn("forecast_yearmonth", F.date_format(F.add_months("curve_start_month", F.col("month_index") - 1), "yyyyMM"))
        .join(performance_monthly, group_columns, "left")
    )
    for metric in metrics:
        performance = (performance.fillna({f"{metric}_neighbor_count": 0, f"{metric}_observed_neighbor_count": 0, f"{metric}_filled_neighbor_count": 0})
                       .withColumn(f"{metric}_actual", F.lit(None).cast("double"))
                       .withColumn(f"{metric}_actual_filled", F.lit(None).cast("double"))
                       .withColumn(f"{metric}_actual_is_filled", F.lit(False)))

    audit_keys = ["_target_id", "neighbor_type", "own_status", "_neighbor_source_key"]
    commercial_availability = (
        neighbor_curves.withColumn("_neighbor_source_key", _normalized(F.col("matched_source_name")))
        .groupBy(*audit_keys).agg(F.min("curve_start_month").alias("first_observed_source_month"),
                                 F.max("last_observed_source_month").alias("last_observed_source_month"),
                                 F.max("source_months_observed").cast("long").alias("source_months_observed"),
                                 F.countDistinct("month_index").alias("curve_months_available"))
    )
    perf_availability = (
        neighbor_perf.withColumn("neighbor_type", F.lit("eilers"))
        .withColumn("_neighbor_source_key", _normalized(F.col("matched_source_name")))
        .groupBy(*audit_keys).agg(F.min("_perf_month").alias("first_observed_source_month"),
                                 F.max("_perf_month").alias("last_observed_source_month"),
                                 F.countDistinct("_perf_month").alias("source_months_observed"),
                                 F.least(F.max("month_index"), F.lit(horizon)).cast("long").alias("curve_months_available"))
    )
    availability = commercial_availability.unionByName(perf_availability)
    neighbors = (
        all_neighbors.drop("own_status").join(channels.select("_target_id", "revenue_type", "own_status"), "_target_id", "inner")
        .where((F.col("neighbor_type") == "eilers") | ((F.col("neighbor_type") == "sales") & (F.col("revenue_type") == "sales"))
               | ((F.col("neighbor_type") == "rental") & (F.col("revenue_type") == "lease")))
        .join(availability, audit_keys, "left")
        .withColumn("has_asof_source_data", F.col("source_months_observed").isNotNull())
        .fillna({"source_months_observed": 0, "curve_months_available": 0})
        .withColumn("distance_cutoff", F.lit(values["distance_cutoff"]))
        .withColumn("minimum_neighbors", F.lit(values["minimum_neighbors"]))
        .withColumn("forecast_months", F.lit(horizon))
    )
    retained_counts = neighbors.groupBy("_target_id", "revenue_type", "neighbor_type").agg(
        F.count("neighbor_game").alias("retained_neighbor_count"),
        F.sum(F.col("has_asof_source_data").cast("long")).alias("neighbors_with_asof_history"))
    demand_counts = retained_counts.where(F.col("neighbor_type").isin("sales", "rental")).drop("neighbor_type")
    demand = demand.join(demand_counts, ["_target_id", "revenue_type"], "left").fillna({"retained_neighbor_count": 0, "neighbors_with_asof_history": 0})

    def status_expression(predicted, retained, available):
        return (F.when(F.col("prediction_date").isNull(), F.lit("missing_prediction_date"))
                .when(retained == 0, F.lit("no_retained_neighbors"))
                .when(available == 0, F.lit("no_historical_source_data"))
                .when(~predicted, F.lit("no_usable_neighbor_values_for_month"))
                .otherwise(F.lit("ready")))

    demand = (demand.withColumn("forecast_eligibility_status", status_expression(F.col("forecast_value").isNotNull(), F.col("retained_neighbor_count"), F.col("neighbors_with_asof_history")))
              .withColumn("actuals_attached", F.lit(False)).withColumn("actual_status", F.lit("not_attached")))
    sales_order = Window.partitionBy("_target_id", "revenue_type").orderBy("month_index")
    demand = demand.withColumn("sales_cumulative_decreases", F.coalesce(
        (F.col("revenue_type") == "sales") & (F.col("forecast_value") < F.lag("forecast_value").over(sales_order)), F.lit(False)))
    perf_counts = retained_counts.where(F.col("neighbor_type") == "eilers").drop("neighbor_type")
    performance = performance.join(perf_counts, ["_target_id", "revenue_type"], "left").fillna({"retained_neighbor_count": 0, "neighbors_with_asof_history": 0})
    any_prediction = F.lit(False)
    for metric in metrics:
        any_prediction = any_prediction | F.col(f"{metric}_mean").isNotNull()
    performance = (performance.withColumn("forecast_eligibility_status", status_expression(any_prediction, F.col("retained_neighbor_count"), F.col("neighbors_with_asof_history")))
                   .withColumn("actuals_attached", F.lit(False)).withColumn("actual_status", F.lit("not_attached")))
    frames = {}
    fits = None
    try:
        fit_keys = ["_target_id", "revenue_type"]
        fit_schema = ("struct<linear:array<double>,poly2:array<double>,log:array<double>,"
                      "fit_months:int,last_fit_month:int,best_model:string,best_r2:double>")
        fit_udf = F.udf(fit_demand_curves, fit_schema)
        fits = (baseline.groupBy(*fit_keys)
                .agg(F.collect_list(F.struct("month_index", "forecast_value")).alias("_points"))
                .select(*fit_keys, fit_udf("_points", F.lit(horizon)).alias("_fit")).cache())
        demand = demand.join(fits, fit_keys, "left")
        for name in ("linear", "poly2", "log"):
            demand = demand.withColumn("forecast_" + name, F.element_at(F.col("_fit." + name), F.col("month_index")))
        demand = demand.drop("_fit")
        frames = {"demand": demand.cache(), "performance": performance.cache(), "neighbors": neighbors.cache()}
        demand_coverage = [row.asDict() for row in frames["demand"].groupBy("revenue_type").agg(
            F.count("month_index").alias("target_months"), F.count("forecast_value").alias("predicted_months"),
            F.countDistinct("target_game_key").alias("target_count"),
            F.countDistinct(F.when(F.col("forecast_value").isNotNull(), F.col("target_game_key"))).alias("targets_with_predictions")).collect()]
        perf_coverage = [row.asDict() for row in frames["performance"].groupBy("own_status").agg(
            F.count("month_index").alias("target_months"),
            *[F.count(f"{metric}_mean").alias(f"{metric}_predicted_months") for metric in metrics]).collect()]
        diagnostics = {"demand_status": [row.asDict() for row in frames["demand"].groupBy("revenue_type", "forecast_eligibility_status").count().collect()],
                       "performance_status": [row.asDict() for row in frames["performance"].groupBy("own_status", "forecast_eligibility_status").count().collect()],
                       "actuals_policy": "not attached; evaluated independently in notebook 07"}
        summaries = (channels.select(*fit_keys, "target_game").join(fits, fit_keys, "left")
                     .select(*fit_keys, "target_game", F.coalesce(F.col("_fit.fit_months"), F.lit(0)).alias("fit_months"),
                             F.col("_fit.last_fit_month").alias("last_fit_month"),
                             F.col("_fit.best_model").alias("best_model"), F.col("_fit.best_r2").alias("best_r2"))
                     .orderBy("target_game", "revenue_type").collect())
        diagnostics["demand_curve_fits"] = [r.asDict() for r in summaries]
        diagnostics["sales_cumulative_declines"] = frames["demand"].agg(
            F.sum(F.col("sales_cumulative_decreases").cast("long")).alias("declining_months"),
            F.countDistinct(F.when(F.col("sales_cumulative_decreases"), F.col("target_game_key"))).alias("affected_targets")).first().asDict()
        config = {**copy.deepcopy(values), "rank_weights": rank_weights, "prediction_date_rule": "saved prediction_date, falling back to game_release_date",
                  "target_calendar": "prediction month + month_index - 1; never derived from target observations",
                  "neighbor_observation_rule": "source month strictly before prediction month; filter before bounds and fills",
                  "neighbor_alignment": "relative to first observed neighbor source month",
                  "target_policy": "every canonical target, both sales and lease populations, regardless of actuals/mappings",
                  "actuals_policy": "excluded from predictions; compatibility columns null/false",
                  "sales_fill": "internal gaps zero; cumulative sum", "rental_fill": "internal gaps forward-fill",
                  "performance_fill": "internal missing metric statistics forward-fill; leading nulls remain",
                  "fill_bounds": "raw benchmarks stop at last available neighbor month; supported fitted demand curves extend to the horizon",
                  "commercial_eligibility": "published by 05 after checking observed commercial months and Eilers availability",
                  "curve_fit": "origin-constrained least squares; fit raw benchmark months only; shift negative minimum over horizon to zero; compare centered R2 on supported months",
                  "bands": "mean +/- 2 population standard deviations; neighbor dispersion, not calibrated intervals",
                  "performance_weighting": "average duplicate rows per game/month/slot bucket, then metric-specific no_of_slots weights"}
        return {"frames": frames, "coverage": {"demand": demand_coverage, "performance": perf_coverage}, "diagnostics": diagnostics, "config": config}
    except BaseException:
        for frame in frames.values():
            frame.unpersist()
        raise
    finally:
        if fits is not None:
            fits.unpersist()


## Durable forecast progress and verified outputs

Self-contained persistence/tracking helpers. Single active manual writer.


In [0]:
import json
import math
from datetime import date, datetime, timezone
from decimal import Decimal

def _now():
    return datetime.now(timezone.utc).isoformat()

def _normal(value):
    if hasattr(value, "asDict"):
        return _normal(value.asDict(recursive=True))
    if isinstance(value, dict):
        return {str(key): _normal(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_normal(item) for item in value]
    if isinstance(value, (datetime, date)):
        return value.isoformat()
    if isinstance(value, Decimal):
        return str(value)
    if hasattr(value, "item"):
        return _normal(value.item())
    if isinstance(value, float) and not math.isfinite(value):
        raise ValueError("Nonfinite values cannot be persisted as canonical metadata/results.")
    return value

def _json(value):
    return json.dumps(_normal(value), sort_keys=True, separators=(",", ":"), allow_nan=False)

def _quote(name):
    return ".".join("`" + part.replace("`", "``") + "`" for part in name.split("."))

def _numeric_metrics(value, prefix=""):
    metrics = {}
    for key, item in value.items():
        name = f"{prefix}/{key}" if prefix else str(key)
        if isinstance(item, dict):
            metrics.update(_numeric_metrics(item, name))
        elif isinstance(item, (int, float)) and not isinstance(item, bool) and math.isfinite(float(item)):
            metrics[name] = float(item)
    return metrics

"""Append-only forecast outputs and durable progress for one manual writer.

Only a verified Delta ledger completion releases work to notebook 07. MLflow
failures are recorded separately and can be repaired without recomputing curves.
No concurrent-writer coordination or destructive reset is implemented here.
"""


import hashlib
import json
import platform
import time
import uuid
from functools import reduce



OUTPUTS = ("demand", "performance", "neighbors")
TRACKING_COLUMNS = {"forecast_mlflow_run_id", "similarity_mlflow_run_id"}
RUN_SCHEMA = """
work_key string, forecast_run_id string, implementation_version string,
forecast_config_hash string, config_json string, configuration_json string,
experiment_id string, experiment_name string, similarity_mlflow_run_id string,
similarity_config_hash string, source_versions_json string,
source_paths_json string, data_revision string, metadata_json string,
status string, processing_status string, tracking_status string,
forecast_mlflow_run_id string, attempt_id string, attempt_count long,
started_at_utc string, updated_at_utc string, completed_at_utc string,
error_type string, error_message string, tracking_error string,
output_signatures_json string, output_locations_json string
"""
ATTEMPT_SCHEMA = """
attempt_id string, work_key string, forecast_run_id string,
attempt_number long, status string, started_at_utc string,
completed_at_utc string, error_type string, error_message string
"""


def _canonical_column(column, data_type):
    """Make nested maps deterministic before Spark serializes each row."""
    from pyspark.sql import functions as F
    from pyspark.sql.types import ArrayType, DoubleType, FloatType, MapType, StructType

    if isinstance(data_type, (FloatType, DoubleType)):
        # Distributed sums can differ in their final binary digits when Spark
        # changes reduction order. Only the integrity fingerprint rounds to
        # twelve significant digits; published forecast values are untouched.
        numeric = F.when(column == 0, F.lit(0.0)).otherwise(column.cast("double"))
        return F.when(column.isNull(), F.lit(None).cast("string")).otherwise(
            F.format_string("%.12g", numeric))
    if isinstance(data_type, MapType):
        value = F.transform(F.array_sort(F.map_keys(column)), lambda key: F.struct(
            key.alias("key"),
            _canonical_column(F.element_at(column, key), data_type.valueType).alias("value")))
        return F.when(column.isNull(), F.lit(None)).otherwise(value)
    if isinstance(data_type, ArrayType):
        return F.transform(column, lambda item: _canonical_column(item, data_type.elementType))
    if isinstance(data_type, StructType):
        value = F.struct(*[
            _canonical_column(column.getField(field.name), field.dataType).alias(field.name)
            for field in sorted(data_type.fields, key=lambda item: item.name)])
        return F.when(column.isNull(), F.lit(None)).otherwise(value)
    return column


def distributed_signature(frame, columns):
    """A bounded-memory, order-independent multiset signature.

    Float/double values use twelve significant digits in this fingerprint only;
    integer, decimal, date and identity fields remain exact. Count, SHA256
    extrema and five independent exact decimal hash-limb sums
    are computed on workers. The driver receives one aggregate row, never a
    global array of hashes or source data. This is an integrity fingerprint,
    rather than a proof that two arbitrary multisets are mathematically equal.
    """
    from pyspark.sql import functions as F

    fields = {field.name: field for field in frame.schema.fields}
    missing = set(columns) - set(fields)
    if missing:
        raise ValueError(f"Output is missing signature columns: {sorted(missing)}")
    canonical = F.struct(*[
        _canonical_column(F.col(name), fields[name].dataType).alias(name)
        for name in sorted(columns)])
    hashed = frame.select(F.sha2(F.to_json(canonical, {
        "ignoreNullFields": "false", "timeZone": "UTC",
        "dateFormat": "yyyy-MM-dd", "timestampFormat": "yyyy-MM-dd'T'HH:mm:ss.SSSXXX",
    }), 256).alias("digest"))
    offsets = (1, 16, 31, 46, 61)
    aggregate = hashed.agg(
        F.count("digest").alias("row_count"), F.min("digest").alias("hash_min"),
        F.max("digest").alias("hash_max"),
        *[F.sum(F.conv(F.substring("digest", offset, 15), 16, 10).cast("decimal(38,0)"))
          .alias(f"sum_{index}") for index, offset in enumerate(offsets)],
    ).first()
    return {
        "row_count": int(aggregate["row_count"]),
        "algorithm": "sha256-float12g-count-decimal-limb-sums-min-max-v2",
        "hash_sums": [str(aggregate[f"sum_{index}"] or 0) for index in range(len(offsets))],
        "hash_min": aggregate["hash_min"], "hash_max": aggregate["hash_max"],
    }


class ForecastStore:
    """Durable API: begin before compute, then publish three Spark DataFrames.

    tables requires demand/performance/neighbors/runs/attempts paths. Metadata
    requires work_key for publish and accepts configuration/config_json,
    forecast_config_hash, experiment_id/name, similarity identity, source_paths,
    source_versions, data_revision, coverage, config_text, environment and
    optional output_keys mapping. Configuration is retained verbatim as both
    config_json and legacy configuration_json for notebook 07 compatibility.
    """

    def __init__(self, spark, tables, mlflow_path, implementation_version):
        missing = set([*OUTPUTS, "runs", "attempts"]) - set(tables)
        if missing:
            raise ValueError(f"Missing forecast tables: {sorted(missing)}")
        self.spark = spark
        self.tables = dict(tables)
        self.mlflow_path = mlflow_path
        self.implementation_version = implementation_version

    def initialize(self):
        for label, schema in [("runs", RUN_SCHEMA), ("attempts", ATTEMPT_SCHEMA)]:
            table = self.tables[label]
            self.spark.sql(f"CREATE TABLE IF NOT EXISTS {_quote(table)} ({schema}) USING DELTA")
            expected = set(self.spark.createDataFrame([], schema=schema).columns)
            missing = expected - set(self.spark.table(table).columns)
            if missing:
                raise ValueError(f"Incompatible forecast ledger {table}: missing {sorted(missing)}")
        return self

    def _merge(self, table, schema, key, record):
        name = "forecast_store_" + uuid.uuid4().hex
        self.spark.createDataFrame([record], schema=schema).createOrReplaceTempView(name)
        try:
            self.spark.sql(
                f"MERGE INTO {_quote(table)} AS target USING `{name}` AS source "
                f"ON target.`{key}` = source.`{key}` "
                "WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *")
        finally:
            self.spark.catalog.dropTempView(name)

    def get_run(self, work_key):
        from pyspark.sql import functions as F

        if not self.spark.catalog.tableExists(self.tables["runs"]):
            return None
        records = (self.spark.table(self.tables["runs"]).where(F.col("work_key") == work_key)
                   .limit(2).collect())
        if len(records) > 1:
            raise RuntimeError(f"Duplicate forecast work records for {work_key}.")
        return records[0].asDict(recursive=True) if records else None

    def _save(self, record):
        record["status"] = record["processing_status"]
        record["updated_at_utc"] = _now()
        self._merge(self.tables["runs"], RUN_SCHEMA, "work_key", record)
        return record

    def _attempt(self, record, status, error=None):
        self._merge(self.tables["attempts"], ATTEMPT_SCHEMA, "attempt_id", {
            "attempt_id": record["attempt_id"], "work_key": record["work_key"],
            "forecast_run_id": record["forecast_run_id"], "attempt_number": record["attempt_count"],
            "status": status, "started_at_utc": record["started_at_utc"],
            "completed_at_utc": None if status == "running" else record.get("completed_at_utc") or _now(),
            "error_type": None if error is None else type(error).__name__,
            "error_message": None if error is None else str(error)[:16000],
        })

    def _metadata(self, metadata):
        metadata = _normal(dict(metadata))
        configuration = (json.loads(metadata["config_json"]) if metadata.get("config_json") is not None
                         else metadata.get("configuration", metadata.get("config", {})))
        metadata["config_json"] = _json(configuration)
        metadata["forecast_config_hash"] = (metadata.get("forecast_config_hash")
                                            or metadata.get("config_hash")
                                            or hashlib.sha256(metadata["config_json"].encode()).hexdigest())
        metadata["source_versions"] = metadata.get("source_versions", configuration.get("source_versions_resolved", {}))
        metadata["source_paths"] = metadata.get("source_paths", configuration.get("source_paths", {}))
        metadata["experiment_id"] = metadata.get("experiment_id", configuration.get("similarity_experiment_id"))
        if not metadata["experiment_id"]:
            raise ValueError("Forecast metadata requires the upstream similarity experiment_id.")
        metadata["experiment_name"] = str(metadata.get("experiment_name") or metadata.get("label")
                                            or metadata["experiment_id"])
        metadata["presentation"] = metadata.get("presentation", {})
        metadata["data_revision"] = str(metadata.get("data_revision", configuration.get(
            "forecast_data_revision", configuration.get("data_revision", ""))))
        for key in ["similarity_mlflow_run_id", "similarity_config_hash"]:
            metadata[key] = metadata.get(key, configuration.get(key))
        return metadata

    def _assert_identity(self, record, metadata):
        for name in ["forecast_config_hash", "config_json", "experiment_id", "similarity_config_hash", "data_revision"]:
            if record.get(name) != metadata.get(name):
                raise ValueError(f"Forecast work key already belongs to different {name}.")
        if record["implementation_version"] != self.implementation_version:
            raise ValueError("Forecast work key already belongs to another implementation version.")
        for key in ["source_versions", "source_paths"]:
            if record[key + "_json"] != _json(metadata[key]):
                raise ValueError(f"Forecast work key already belongs to different {key}.")

    def begin(self, work_key, metadata):
        if not isinstance(work_key, str) or not work_key.strip():
            raise ValueError("A nonempty semantic forecast work_key is required.")
        metadata = self._metadata({**metadata, "work_key": work_key})
        record = self.get_run(work_key)
        if record:
            self._assert_identity(record, metadata)
            prior_metadata = json.loads(record["metadata_json"])
            metadata["requested_presentation"] = metadata["presentation"]
            metadata["presentation"] = prior_metadata.get("presentation", metadata["presentation"])
            audit_metadata = {**prior_metadata, "presentation": metadata["presentation"],
                              "requested_presentation": metadata["requested_presentation"]}
            if _json(audit_metadata) != record["metadata_json"]:
                record["metadata_json"] = _json(audit_metadata)
                self._save(record)
            previous_tracking = record.get("similarity_mlflow_run_id")
            incoming_tracking = metadata.get("similarity_mlflow_run_id")
            if previous_tracking and incoming_tracking and previous_tracking != incoming_tracking:
                raise ValueError("The similarity experiment already references a different MLflow run.")
            metadata["similarity_mlflow_run_id"] = previous_tracking or incoming_tracking
            if not previous_tracking and incoming_tracking:
                # Upstream tracking may have recovered after these curves were
                # completed. Enrich lineage without changing semantic identity
                # or recalculating forecasts.
                old_metadata = json.loads(record["metadata_json"])
                old_metadata["similarity_mlflow_run_id"] = incoming_tracking
                record.update(similarity_mlflow_run_id=incoming_tracking,
                              metadata_json=_json(old_metadata), tracking_status="pending")
                self._save(record)
            if record["processing_status"] == "complete":
                self._verify_all(record)
                self._attempt(record, "complete")
                return {**self.repair_tracking(work_key), "presentation": metadata["presentation"],
                        "skip_compute": True}
            if record["processing_status"] == "running":
                self._attempt(record, "interrupted")
            metadata = {**json.loads(record["metadata_json"]), **metadata}
            metadata["requested_experiment_name"] = metadata["experiment_name"]
            metadata["experiment_name"] = record["experiment_name"]
        else:
            record = {
                "work_key": work_key,
                "forecast_run_id": "fcst_" + hashlib.sha256(work_key.encode()).hexdigest()[:32],
                "implementation_version": self.implementation_version,
                "attempt_count": 0, "completed_at_utc": None,
                "forecast_mlflow_run_id": None, "tracking_status": "pending",
                "output_signatures_json": None, "output_locations_json": None,
            }
        record.update(
            forecast_config_hash=metadata["forecast_config_hash"], config_json=metadata["config_json"],
            configuration_json=metadata["config_json"], experiment_id=metadata["experiment_id"],
            experiment_name=metadata["experiment_name"], similarity_config_hash=metadata["similarity_config_hash"],
            similarity_mlflow_run_id=metadata["similarity_mlflow_run_id"],
            source_versions_json=_json(metadata["source_versions"]), source_paths_json=_json(metadata["source_paths"]),
            data_revision=metadata["data_revision"], metadata_json=_json(metadata),
            processing_status="running", attempt_id="attempt_" + uuid.uuid4().hex,
            attempt_count=int(record["attempt_count"]) + 1, started_at_utc=_now(),
            error_type=None, error_message=None, tracking_error=None,
        )
        self._save(record)
        self._attempt(record, "running")
        return {**self._ensure_tracking_run(record), "presentation": metadata["presentation"],
                "skip_compute": False}

    def _ensure_tracking_run(self, record):
        try:
            from mlflow.tracking import MlflowClient
            from mlflow.entities import ViewType

            client = MlflowClient()
            if record.get("forecast_mlflow_run_id"):
                run = client.get_run(record["forecast_mlflow_run_id"])
                if run.info.lifecycle_stage != "active":
                    raise RuntimeError("The MLflow forecast run was deleted; restore it to repair tracking.")
                return record
            experiment = client.get_experiment_by_name(self.mlflow_path)
            experiment_id = experiment.experiment_id if experiment else client.create_experiment(self.mlflow_path)
            matches = client.search_runs(
                [experiment_id], filter_string=f"tags.forecast_run_id = '{record['forecast_run_id']}'",
                run_view_type=ViewType.ALL, max_results=2)
            if len(matches) > 1:
                raise RuntimeError("Multiple MLflow runs own this forecast identity.")
            if matches:
                if matches[0].info.lifecycle_stage != "active":
                    raise RuntimeError("The MLflow forecast run was deleted; restore it to repair tracking.")
                run_id = matches[0].info.run_id
            else:
                run_id = client.create_run(experiment_id, tags={
                    "mlflow.runName": f"{record['experiment_name']}_{record['forecast_run_id'][-8:]}",
                    "pipeline_stage": "forecast", "forecast_run_id": record["forecast_run_id"],
                    "forecast_work_key": record["work_key"], "forecast_config_hash": record["forecast_config_hash"],
                    "similarity_experiment_id": record["experiment_id"],
                    "implementation_version": self.implementation_version,
                }).info.run_id
            record.update(forecast_mlflow_run_id=run_id, tracking_status="pending", tracking_error=None)
        except Exception as error:
            record.update(tracking_status="failed", tracking_error=f"{type(error).__name__}: {error}"[:16000])
        return self._save(record)

    def _stored(self, label, record):
        from pyspark.sql import functions as F

        table = self.tables[label]
        if not self.spark.catalog.tableExists(table):
            return None
        return self.spark.table(table).where(F.col("forecast_run_id") == record["forecast_run_id"])

    def _keys(self, label, frame, metadata):
        configured = metadata.get("output_keys", {}).get(label)
        if configured:
            return list(configured)
        base = ["target_game_key", "revenue_type"]
        if label == "demand":
            return [*base, "month_index"]
        if label == "performance":
            return [*base, "own_status", "month_index"]
        source = next((key for key in ["_neighbor_source_key", "matched_source_name", "neighbor_game"]
                       if key in frame.columns), "_neighbor_source_key")
        return [*base, "own_status", "neighbor_type", source]

    def _verify_keys(self, frame, keys, label):
        from pyspark.sql import functions as F

        missing = set(keys) - set(frame.columns)
        if missing:
            raise ValueError(f"{label} lacks required output keys: {sorted(missing)}")
        nulls = reduce(lambda a, b: a | b, [F.col(key).isNull() for key in keys])
        if frame.where(nulls).limit(1).count():
            raise ValueError(f"{label} contains null output keys {keys}.")
        if frame.groupBy(*keys).count().where(F.col("count") > 1).limit(1).count():
            raise ValueError(f"{label} contains duplicate output keys {keys}.")

    def _verify_output(self, label, record, expected):
        from pyspark.sql import functions as F

        frame = self._stored(label, record)
        if frame is None:
            raise RuntimeError(f"The {label} output table does not exist.")
        self._verify_keys(frame, expected["key_columns"], label)
        signature = distributed_signature(frame, expected["columns"])
        signature.update(columns=expected["columns"], key_columns=expected["key_columns"])
        if signature != expected:
            raise RuntimeError(f"The {label} saved count/content differs from its immutable expected signature.")
        for name in sorted(TRACKING_COLUMNS):
            conflicting = frame.where(
                F.col(name).isNotNull() & ~F.col(name).eqNullSafe(F.lit(record.get(name))))
            if conflicting.limit(1).count():
                raise RuntimeError(f"The {label} output references a different {name}.")

    def _verify_all(self, record):
        if not record.get("output_signatures_json"):
            raise RuntimeError("Completed forecast has no expected output signatures.")
        signatures = json.loads(record["output_signatures_json"])
        if set(signatures) != set(OUTPUTS):
            raise RuntimeError("Forecast signature manifest does not contain all three outputs.")
        for label in OUTPUTS:
            self._verify_output(label, record, signatures[label])

    def publish(self, frames, metadata):
        from pyspark.sql import functions as F

        if set(frames) != set(OUTPUTS):
            raise ValueError(f"publish requires exactly these frames: {OUTPUTS}")
        work_key = metadata.get("work_key")
        record = self.get_run(work_key)
        if record is None:
            raise ValueError("Call begin before computing/publishing a forecast.")
        previous_metadata = json.loads(record["metadata_json"])
        merged = self._metadata({**previous_metadata, **metadata})
        self._assert_identity(record, merged)
        merged["requested_presentation"] = merged["presentation"]
        merged["presentation"] = previous_metadata.get("presentation", merged["presentation"])
        merged["requested_experiment_name"] = merged["experiment_name"]
        merged["experiment_name"] = record["experiment_name"]
        if record["processing_status"] == "complete":
            self._verify_all(record)
            return self.repair_tracking(work_key)
        if record["processing_status"] != "running":
            raise ValueError("Call begin again before retrying a failed forecast.")
        prepared, expected = {}, {}
        for label, frame in frames.items():
            for name, value in {
                "forecast_run_id": record["forecast_run_id"], "forecast_work_key": work_key,
                "forecast_config_hash": record["forecast_config_hash"],
                "forecast_mlflow_run_id": record["forecast_mlflow_run_id"],
                "experiment_id": record["experiment_id"], "experiment_name": record["experiment_name"],
                "similarity_mlflow_run_id": record["similarity_mlflow_run_id"],
                "similarity_config_hash": record["similarity_config_hash"],
                "data_revision": record["data_revision"],
            }.items():
                frame = frame.withColumn(name, F.lit(value).cast("string"))
            keys = self._keys(label, frame, merged)
            self._verify_keys(frame, keys, label)
            columns = sorted(set(frame.columns) - TRACKING_COLUMNS)
            expected[label] = {**distributed_signature(frame, columns), "columns": columns, "key_columns": keys}
            prepared[label] = frame
        if record.get("output_signatures_json") and json.loads(record["output_signatures_json"]) != expected:
            raise RuntimeError("Retry produced different output content/keys for the same forecast work key.")
        record.update(output_signatures_json=_json(expected), metadata_json=_json(merged))
        # Persist expected content for EVERY output before committing the first.
        self._save(record)
        locations = {}
        for label in OUTPUTS:
            existing = self._stored(label, record)
            if existing is None or not existing.limit(1).count():
                (prepared[label].write.format("delta").mode("append").option("mergeSchema", "true")
                 .option("txnAppId", f"forecast:{record['forecast_run_id']}:{label}").option("txnVersion", 0)
                 .option("userMetadata", _json({"forecast_run_id": record["forecast_run_id"],
                                               "work_key": work_key, "output": label}))
                 .saveAsTable(self.tables[label]))
            self._verify_output(label, record, expected[label])
            commit = (self.spark.sql(f"DESCRIBE HISTORY {_quote(self.tables[label])}")
                      .where((F.get_json_object(F.col("userMetadata"), "$.forecast_run_id") == record["forecast_run_id"])
                             & (F.get_json_object(F.col("userMetadata"), "$.output") == label))
                      .agg(F.min("version").alias("v")).first()["v"])
            version = None if commit is None else int(commit)
            locations[label] = {"table": self.tables[label], "delta_commit_version": version,
                                "delta_version": version, "row_count": expected[label]["row_count"]}
        self._verify_all(record)
        record.update(output_locations_json=_json(locations), processing_status="complete",
                      completed_at_utc=_now(), error_type=None, error_message=None)
        self._save(record)
        self._attempt(record, "complete")
        return self.repair_tracking(work_key)

    def repair_tracking(self, work_key):
        record = self.get_run(work_key)
        if record is None:
            raise ValueError(f"Unknown forecast work key: {work_key}")
        if record["processing_status"] != "complete" or record["tracking_status"] == "complete":
            return record
        record = self._ensure_tracking_run(record)
        if not record.get("forecast_mlflow_run_id"):
            return record
        try:
            import mlflow
            from mlflow.tracking import MlflowClient
            from mlflow.entities import Param, Metric
            from pyspark.sql import functions as F

            self._verify_all(record)
            client, run_id = MlflowClient(), record["forecast_mlflow_run_id"]
            metadata, configuration = json.loads(record["metadata_json"]), json.loads(record["config_json"])
            client.log_dict(run_id, configuration, "forecast_configuration.json")
            client.log_dict(run_id, json.loads(record["output_locations_json"]), "results/delta_outputs.json")
            if metadata.get("config_text") is not None:
                client.log_text(run_id, metadata["config_text"], "configuration/batch.yaml")
            for key, artifact in [("coverage", "results/coverage.json"),
                                  ("diagnostics", "results/diagnostics.json"),
                                  ("algorithm", "algorithm.json")]:
                if metadata.get(key) is not None:
                    client.log_dict(run_id, metadata[key], artifact)
            client.log_dict(run_id, {
                "python": platform.python_version(), "spark": self.spark.version,
                "mlflow": mlflow.__version__, "implementation_version": self.implementation_version,
                **metadata.get("environment", {}),
            }, "environment.json")
            settings = configuration.get("forecast", configuration)
            params = {"forecast_config_hash": record["forecast_config_hash"],
                      "similarity_experiment_id": record["experiment_id"],
                      "similarity_config_hash": record["similarity_config_hash"],
                      "data_revision": record["data_revision"],
                      "implementation_version": self.implementation_version}
            spec = configuration.get("named_experiment")
            if spec:
                params.update(named_experiment=spec["name"], demand_rank_weights=_json(spec["rank_weights"]),
                              train_bias_correction=spec["train_bias_correction"])
            for key in ["distance_method", "parent_game_process", "distance_cutoff", "minimum_neighbors",
                        "horizon_months", "prediction_date_rule", "neighbor_observation_rule"]:
                params[key] = settings.get(key, configuration.get(key))
            similarity = metadata.get("similarity_settings", {})
            for key in ["distance_method", "parent_game_process"]:
                if params.get(key) is None:
                    params[key] = similarity.get(key)
            for group, value in similarity.get("raw_weights", {}).items():
                params[f"raw_weight_{group}"] = value
            for group, value in similarity.get("normalized_weights", {}).items():
                params[f"weight_{group}"] = value
            for source, limit in similarity.get("neighbor_limits", {}).items():
                params[f"similarity_n_{source}_neighbors"] = limit
            for key in ["slot_counts", "performance_metrics"]:
                if key in settings:
                    params[key] = _json(settings[key])
            for group, value in configuration.get("feature_group_weights", {}).items():
                params[f"weight_{group}"] = value
            for source, version in json.loads(record["source_versions_json"]).items():
                params[f"source_version_{source}"] = version
            params = {key: str(value) for key, value in params.items() if value is not None}
            previous = client.get_run(run_id).data.params
            conflicts = [key for key, value in params.items() if key in previous and previous[key] != value]
            if conflicts:
                raise RuntimeError(f"Immutable forecast tracking parameters differ: {sorted(conflicts)}")
            signatures = json.loads(record["output_signatures_json"])
            metrics = {f"{label}_rows": float(signatures[label]["row_count"]) for label in OUTPUTS}
            metrics.update(_numeric_metrics(metadata.get("coverage", {}), "coverage"))
            timestamp = int(time.time() * 1000)
            client.log_batch(
                run_id, params=[Param(key, value) for key, value in params.items() if key not in previous],
                metrics=[Metric(key, value, timestamp, 0) for key, value in metrics.items()], synchronous=True)
            if record.get("similarity_mlflow_run_id"):
                client.set_tag(run_id, "upstream_similarity_mlflow_run_id", record["similarity_mlflow_run_id"])
            # Null lineage may be enriched after a tracking outage. It is the
            # only mutable output content and is excluded from signatures.
            for label in OUTPUTS:
                patch_name = "forecast_tracking_" + uuid.uuid4().hex
                self.spark.createDataFrame(
                    [(record["forecast_run_id"], run_id, record.get("similarity_mlflow_run_id"))],
                    "forecast_run_id string, forecast_mlflow_run_id string, similarity_mlflow_run_id string"
                ).createOrReplaceTempView(patch_name)
                try:
                    bad = self._stored(label, record).where(
                        F.col("forecast_mlflow_run_id").isNotNull() & (F.col("forecast_mlflow_run_id") != run_id))
                    if bad.limit(1).count():
                        raise RuntimeError(f"{label} already belongs to another MLflow forecast run.")
                    self.spark.sql(
                        f"MERGE INTO {_quote(self.tables[label])} AS target USING `{patch_name}` AS source "
                        "ON target.forecast_run_id = source.forecast_run_id WHEN MATCHED AND "
                        "(target.forecast_mlflow_run_id IS NULL OR "
                        "(target.similarity_mlflow_run_id IS NULL AND source.similarity_mlflow_run_id IS NOT NULL)) "
                        "THEN UPDATE SET target.forecast_mlflow_run_id = source.forecast_mlflow_run_id, "
                        "target.similarity_mlflow_run_id = coalesce(target.similarity_mlflow_run_id, source.similarity_mlflow_run_id)")
                finally:
                    self.spark.catalog.dropTempView(patch_name)
                patched = self._stored(label, record)
                for name in TRACKING_COLUMNS:
                    if record.get(name) is not None and patched.where(
                        ~F.col(name).eqNullSafe(F.lit(record[name]))
                    ).limit(1).count():
                        raise RuntimeError(f"The {label} {name} enrichment was not preserved.")
            self._verify_all(record)
            client.set_tag(run_id, "forecast_save_status", "complete")
            client.set_terminated(run_id, status="FINISHED")
            record.update(tracking_status="complete", tracking_error=None)
        except Exception as error:
            record.update(tracking_status="failed", tracking_error=f"{type(error).__name__}: {error}"[:16000])
        return self._save(record)

    def fail(self, work_key, error, attempt_id=None):
        record = self.get_run(work_key)
        if record is None:
            raise ValueError(f"Unknown forecast work key: {work_key}")
        if attempt_id is not None and attempt_id != record["attempt_id"]:
            raise ValueError("Failure belongs to an earlier forecast attempt.")
        if record["processing_status"] == "complete":
            return record
        record.update(processing_status="failed", error_type=type(error).__name__, error_message=str(error)[:16000])
        self._save(record)
        self._attempt(record, "failed", error)
        if record.get("forecast_mlflow_run_id"):
            try:
                from mlflow.tracking import MlflowClient

                client = MlflowClient()
                client.set_tag(record["forecast_mlflow_run_id"], "failure_type", type(error).__name__)
                client.set_terminated(record["forecast_mlflow_run_id"],
                                      status="KILLED" if isinstance(error, KeyboardInterrupt) else "FAILED")
            except Exception:
                pass
        return record

    def status_df(self):
        return self.spark.table(self.tables["runs"])


## Discover completed 05 generations and plan pending work


In [0]:
from datetime import datetime, timezone
from uuid import uuid4
from pyspark.sql import functions as F

FORECAST_CONFIG = load_forecast_config(CONFIG_PATH)
preflight_forecast_tables(spark, FORECAST_CONFIG)
FORECAST_OUTPUT_TABLES = FORECAST_CONFIG["outputs"]
FORECAST_STORE = ForecastStore(spark, FORECAST_OUTPUT_TABLES,
    FORECAST_CONFIG["tracking"]["experiment_path"], FORECAST_IMPLEMENTATION_VERSION).initialize()
FORECAST_BATCH_ID = "batch_" + uuid4().hex
FORECAST_BATCH_STARTED_AT_UTC = datetime.now(timezone.utc).isoformat()


def discover_completed_similarity(spark, config, selection_history):
    """Freeze the published upstream table once; tracking availability is optional."""
    common = config["common"]
    result_path = common["outputs"]["like_games_table"]
    runs_path = common["outputs"]["runs_table"]
    for path in [result_path, runs_path]:
        if not spark.catalog.tableExists(path):
            raise ValueError(f"Required new-05 table is missing: {path}. Run the updated notebook 05 first.")
    version = int(spark.sql(f"DESCRIBE HISTORY {quoted_table(result_path)}").agg(F.max("version").alias("v")).first()["v"])
    saved = spark.read.option("versionAsOf", version).table(result_path)
    required = {"experiment_id", "join_game_name", "similarity_work_key", "similarity_config_hash", "similarity_result_hash"}
    missing = required - set(saved.columns)
    if missing:
        raise ValueError(f"Like-games table is not from new notebook 05: missing {sorted(missing)}")
    ledger = spark.table(runs_path)
    ledger_required = {"implementation_version", "work_key", "experiment_id", "experiment_name", "status", "result_table",
                       "result_row_count", "config_hash", "result_hash", "config_json", "similarity_mlflow_run_id"}
    if not ledger_required.issubset(ledger.columns):
        raise ValueError("Similarity completion records use an incompatible schema; run new 05 first.")
    completed = ledger.where((F.col("status") == "complete") & (F.col("result_table") == result_path)
                             & (F.col("implementation_version") == SIMILARITY_IMPLEMENTATION_VERSION))
    records = [row.asDict(recursive=True) for row in completed.select(*sorted(ledger_required)).collect()]
    data = common["data"]
    ai_version = resolve_similarity_source_version(spark, common)
    matching = []
    for row in records:
        saved_config = json.loads(row["config_json"])
        if (saved_config.get("selection_history") != selection_history or
            saved_config.get("data_revision") != data["revision"] or
            saved_config.get("source_table") != data["ai_attributes_table"] or
            saved_config.get("source_table_version") != ai_version):
            continue
        if data["prediction_as_of_date"] is None:
            if saved_config.get("prediction_as_of_source") != "assumed_recent":
                continue
        elif (saved_config.get("prediction_as_of_source") != "configured" or
              saved_config.get("prediction_as_of_date") != data["prediction_as_of_date"]):
            continue
        matching.append(row)
    records = matching
    selected_ids = config["selection"]["similarity_experiment_ids"]
    if selected_ids:
        available = {row["experiment_id"] for row in records}
        unknown = set(selected_ids) - available
        if unknown:
            raise ValueError(f"These IDs have no completed generation with the current history settings and snapshots: {sorted(unknown)}")
        records = [row for row in records if row["experiment_id"] in selected_ids]
    records.sort(key=lambda row: (row["experiment_name"], row["experiment_id"]))
    if len({row["experiment_id"] for row in records}) != len(records):
        raise ValueError("Duplicate completed similarity identities were found; resolve upstream history before forecasting.")
    return saved, records, version


def plan_named_forecasts(records, config, source_versions):
    """Pair each enabled YAML experiment with its matching saved 05 generation."""
    by_settings = {}
    for upstream in records:
        settings = json.loads(upstream["config_json"])["similarity"]
        by_settings.setdefault(config_sha256(settings), []).append(upstream)
    plans = []
    missing = []
    for item in config["common"]["experiments"]:
        if not item["enabled"]:
            continue
        spec = item["forecast"]
        matches = by_settings.get(spec["similarity_settings_hash"], [])
        if not matches:
            if not config["selection"]["similarity_experiment_ids"]:
                missing.append(item["name"])
            continue
        if len(matches) != 1:
            raise ValueError(f"Multiple completed 05 generations match {item['name']}. Set forecast.selection.similarity_experiment_ids to the intended ID.")
        upstream = matches[0]
        key, semantic = forecast_work_identity(upstream, config, source_versions, spec)
        plans.append({"upstream": upstream, "work_key": key, "semantic": semantic,
                      "forecast": spec, "experiment_name": item["name"]})
    if missing:
        raise ValueError(f"Run 05 for the enabled configurations missing saved neighbors: {missing}")
    return plans


FORECAST_SOURCE_VERSIONS = resolve_forecast_source_versions(spark, FORECAST_CONFIG)
EXPECTED_SELECTION = neighbor_selection_policy(FORECAST_CONFIG, FORECAST_SOURCE_VERSIONS)
ALL_SAVED_NEIGHBORS_DF, COMPLETED_SIMILARITY, LIKE_GAMES_TABLE_VERSION = discover_completed_similarity(
    spark, FORECAST_CONFIG, EXPECTED_SELECTION)
FORECAST_PLAN = plan_named_forecasts(COMPLETED_SIMILARITY, FORECAST_CONFIG, FORECAST_SOURCE_VERSIONS)
print(f"Forecast batch: {FORECAST_BATCH_ID}")
print(f"Forecast data revision: {FORECAST_CONFIG['data_revision']}; snapshots: {FORECAST_SOURCE_VERSIONS}")
print(f"Required 05 version: {SIMILARITY_IMPLEMENTATION_VERSION}. Run updated 05 first if none are listed.")
print(f"Completed upstream generations: {len(COMPLETED_SIMILARITY)}. Planned forecasts: {len(FORECAST_PLAN)}. One active manual batch at a time.")
print("Predictions use explicit target prediction dates. Evaluation actuals are not loaded into these outputs.")
if FORECAST_PLAN:
    display(spark.createDataFrame([
        (p["experiment_name"], p["forecast"]["rank_weights"], p["upstream"]["experiment_id"], p["work_key"])
        for p in FORECAST_PLAN
    ], "experiment_name string, demand_rank_weights array<double>, experiment_id string, forecast_work_key string"))


## Process missing forecasts

Completed outputs are verified/skipped. Shared source preparation runs only when work is pending.


In [0]:
import json
import platform
import time
import pandas as pd
from pyspark.sql import functions as F

if load_forecast_config(CONFIG_PATH) != FORECAST_CONFIG:
    raise ValueError("The YAML changed after loading. Rerun notebook 06 from the top.")

FORECAST_BATCH_OUTCOMES = []
FORECAST_PENDING = []


def forecast_outcome(plan, record=None, action="failed", error=None):
    record = record or {}
    metadata = json.loads(record.get("metadata_json") or "{}")
    coverage = metadata.get("coverage", {})
    return {"batch_id": FORECAST_BATCH_ID, "experiment_name": record.get("experiment_name", plan["experiment_name"]),
            "experiment_id": plan["upstream"]["experiment_id"], "forecast_run_id": record.get("forecast_run_id"),
            "action": action, "processing_status": record.get("processing_status", "failed"),
            "tracking_status": record.get("tracking_status", "pending"), "attempt_count": record.get("attempt_count"),
            "coverage": json.dumps(coverage, sort_keys=True),
            "error": None if error is None else f"{type(error).__name__}: {error}"}


def coverage_by_population(coverage):
    """Named populations keep monitoring/MLflow keys stable across batches."""
    result = {}
    for label, key in [("demand", "revenue_type"), ("performance", "own_status")]:
        rows = coverage.get(label, [])
        result[label] = ({str(row[key]): {k: v for k, v in row.items() if k != key} for row in rows}
                         if isinstance(rows, list) else rows)
    return result


for plan in FORECAST_PLAN:
    upstream = plan["upstream"]
    metadata = {
        "work_key": plan["work_key"], "configuration": plan["semantic"], "forecast_config_hash": plan["work_key"],
        "experiment_id": upstream["experiment_id"], "experiment_name": plan["experiment_name"],
        "similarity_experiment_name": upstream["experiment_name"],
        "similarity_mlflow_run_id": upstream["similarity_mlflow_run_id"],
        "similarity_config_hash": upstream["config_hash"], "similarity_result_hash": upstream["result_hash"],
        "similarity_settings": json.loads(upstream["config_json"]).get("similarity", {}),
        "source_versions": FORECAST_SOURCE_VERSIONS, "source_paths": FORECAST_CONFIG["sources"],
        "data_revision": FORECAST_CONFIG["data_revision"],
        "batch_id": FORECAST_BATCH_ID, "batch_started_at_utc": FORECAST_BATCH_STARTED_AT_UTC,
        "like_games_table": FORECAST_CONFIG["common"]["outputs"]["like_games_table"],
        "like_games_table_version": LIKE_GAMES_TABLE_VERSION,
        "config_text": FORECAST_CONFIG["common"]["config_text"],
        "yaml_sha256": FORECAST_CONFIG["common"]["yaml_sha256"],
        "environment": {"python": platform.python_version(), "spark": spark.version},
        "presentation": {"lease_curve_label": FORECAST_CONFIG["settings"]["lease_curve_label"]},
    }
    try:
        record = FORECAST_STORE.begin(plan["work_key"], metadata)
        if record["skip_compute"]:
            FORECAST_BATCH_OUTCOMES.append(forecast_outcome(plan, record, "skipped_complete"))
        else:
            metadata["experiment_name"] = record["experiment_name"]
            metadata["presentation"] = json.loads(record["metadata_json"]).get("presentation", metadata["presentation"])
            FORECAST_PENDING.append((plan, metadata, record))
    except Exception as error:
        FORECAST_BATCH_OUTCOMES.append(forecast_outcome(plan, action="failed_to_begin", error=error))

shared_forecast_sources = None
try:
    if FORECAST_PENDING:
        started = time.monotonic()
        try:
            source_frames = {key: spark.read.option("versionAsOf", FORECAST_SOURCE_VERSIONS[key]).table(path)
                             for key, path in FORECAST_CONFIG["sources"].items()}
            shared_forecast_sources = prepare_forecast_sources(spark, source_frames, FORECAST_CONFIG["settings"])
            shared_preparation_seconds = time.monotonic() - started
            print(f"Prepared shared Sales/Rental/Eilers sources once for {len(FORECAST_PENDING)} pending experiments.")
        except BaseException as error:
            for plan, metadata, record in FORECAST_PENDING:
                try:
                    failed = FORECAST_STORE.fail(plan["work_key"], error, record["attempt_id"])
                    FORECAST_BATCH_OUTCOMES.append(forecast_outcome(plan, failed, "shared_preparation_failed", error))
                except Exception as status_error:
                    FORECAST_BATCH_OUTCOMES.append(forecast_outcome(plan, record, "status_write_failed", status_error))
            if not isinstance(error, Exception):
                raise

    if shared_forecast_sources is not None:
        for index, (plan, metadata, record) in enumerate(FORECAST_PENDING, 1):
            output_frames = {}
            upstream = plan["upstream"]
            print(f"[{index}/{len(FORECAST_PENDING)}] {record['experiment_name']} | {record['forecast_run_id']}")
            started = time.monotonic()
            try:
                generation_frame = ALL_SAVED_NEIGHBORS_DF.where(F.col("experiment_id") == upstream["experiment_id"])
                check = generation_frame.agg(
                    F.count(F.lit(1)).alias("rows"), F.countDistinct("join_game_name").alias("games"),
                    F.sum(F.when(~F.col("similarity_work_key").eqNullSafe(F.lit(upstream["work_key"]))
                        | ~F.col("similarity_config_hash").eqNullSafe(F.lit(upstream["config_hash"]))
                        | ~F.col("similarity_result_hash").eqNullSafe(F.lit(upstream["result_hash"])), 1).otherwise(0)).alias("bad_lineage"),
                ).first()
                if check["rows"] != upstream["result_row_count"] or check["games"] != check["rows"] or check["bad_lineage"]:
                    raise ValueError("Saved upstream target count, canonical keys, or generation lineage differs from its completed record.")
                similarity_settings = json.loads(upstream["config_json"]).get("similarity", {})
                context = {
                    "experiment_id": upstream["experiment_id"], "experiment_name": record["experiment_name"],
                    "experiment_readable_name": record["experiment_name"], "forecast_run_id": record["forecast_run_id"],
                    "forecast_config_hash": plan["work_key"], "config_hash": plan["work_key"],
                    "forecast_mlflow_run_id": record.get("forecast_mlflow_run_id"),
                    "similarity_mlflow_run_id": upstream["similarity_mlflow_run_id"],
                    "similarity_config_hash": upstream["config_hash"], "similarity_result_hash": upstream["result_hash"],
                    "distance_method": similarity_settings.get("distance_method"),
                    "feature_group_weights": similarity_settings.get("raw_weights", {}),
                    "parent_game_process": similarity_settings.get("parent_game_process"),
                }
                run_settings = {**FORECAST_CONFIG["settings"], **metadata["presentation"]}
                generated = generate_forecast(spark, generation_frame, shared_forecast_sources, context, run_settings,
                                              rank_weights=plan["forecast"]["rank_weights"])
                output_frames = {key: frame.cache() for key, frame in generated["frames"].items()}
                metadata.update(coverage=coverage_by_population(generated["coverage"]), diagnostics=generated.get("diagnostics", {}),
                                algorithm=generated.get("config", {}), shared_preparation_seconds=shared_preparation_seconds,
                                generation_seconds=time.monotonic() - started)
                completed = FORECAST_STORE.publish(output_frames, metadata)
                FORECAST_BATCH_OUTCOMES.append(forecast_outcome(plan, completed, "processed"))
            except BaseException as error:
                try:
                    failed = FORECAST_STORE.fail(plan["work_key"], error, record["attempt_id"])
                    FORECAST_BATCH_OUTCOMES.append(forecast_outcome(plan, failed, "failed", error))
                except Exception as status_error:
                    FORECAST_BATCH_OUTCOMES.append(forecast_outcome(plan, record, "status_write_failed", status_error))
                if not isinstance(error, Exception):
                    raise
            finally:
                for frame in output_frames.values():
                    frame.unpersist()
finally:
    if shared_forecast_sources is not None:
        release_forecast_sources(shared_forecast_sources)

forecast_batch_summary_pdf = pd.DataFrame(FORECAST_BATCH_OUTCOMES, columns=[
    "batch_id", "experiment_name", "experiment_id", "forecast_run_id", "action", "processing_status", "tracking_status", "attempt_count", "coverage", "error"])
display(forecast_batch_summary_pdf)
failures = [row for row in FORECAST_BATCH_OUTCOMES if row["error"] is not None]
tracking_pending = [row for row in FORECAST_BATCH_OUTCOMES if row["tracking_status"] != "complete"]
print(f"Discovered={len(FORECAST_PLAN)}, processed={sum(x['action'] == 'processed' for x in FORECAST_BATCH_OUTCOMES)}, "
      f"skipped={sum(x['action'] == 'skipped_complete' for x in FORECAST_BATCH_OUTCOMES)}, failed={len(failures)}")
if tracking_pending:
    print(f"Tracking needs attention for {len(tracking_pending)} item(s). Verified predictions remain complete; rerun to repair logging without recomputing them.")
if failures:
    raise RuntimeError("Forecast batch has failed experiments. Errors are saved above and in forecast_runs; rerun to retry pending work.")
if not FORECAST_PENDING:
    print("Nothing new to forecast. Completed outputs were verified and skipped.")


## Forecast processing and coverage monitoring

Read-only status and attempt history. No forecast recalculation.


In [0]:
from pyspark.sql import functions as F

forecast_monitor_df = FORECAST_STORE.status_df().withColumn(
    "duration_seconds", F.coalesce(F.col("completed_at_utc").cast("timestamp"),
        F.when(F.col("processing_status") == "running", F.current_timestamp())
         .otherwise(F.col("updated_at_utc").cast("timestamp"))).cast("double")
        - F.col("started_at_utc").cast("timestamp").cast("double"))
for channel in ["sales", "lease"]:
    for name in ["target_months", "predicted_months", "target_count", "targets_with_predictions"]:
        forecast_monitor_df = forecast_monitor_df.withColumn(f"{channel}_{name}", F.get_json_object(
            "metadata_json", f"$.coverage.demand.{channel}.{name}").cast("long"))
    forecast_monitor_df = forecast_monitor_df.withColumn(f"{channel}_coverage", F.when(
        F.col(f"{channel}_target_months") > 0, F.col(f"{channel}_predicted_months") / F.col(f"{channel}_target_months")))
display(forecast_monitor_df.select(
    "experiment_name", "experiment_id", "forecast_run_id", "data_revision",
    "processing_status", "tracking_status", "attempt_count", "duration_seconds",
    "sales_target_count", "sales_targets_with_predictions", "sales_coverage", "lease_targets_with_predictions", "lease_coverage",
    F.get_json_object("metadata_json", "$.coverage.performance").alias("performance_coverage"),
    "error_message", "tracking_error", "forecast_mlflow_run_id", "source_versions_json", "updated_at_utc",
).orderBy(F.desc("updated_at_utc")).limit(FORECAST_CONFIG["monitoring"]["history_rows"]))
display(forecast_monitor_df.orderBy(F.desc("updated_at_utc")).select("experiment_name", "forecast_run_id",
    F.get_json_object("metadata_json", "$.diagnostics").alias("diagnostics")
).limit(FORECAST_CONFIG["monitoring"]["history_rows"]))
display(FORECAST_STORE.status_df().groupBy("processing_status", "tracking_status").count())
display(spark.table(FORECAST_OUTPUT_TABLES["attempts"]).orderBy(F.desc("started_at_utc")).limit(
    FORECAST_CONFIG["monitoring"]["history_rows"]))
print(f"Persistent forecast status: {FORECAST_OUTPUT_TABLES['runs']}")
print(f"Persistent attempt history: {FORECAST_OUTPUT_TABLES['attempts']}")
print("These are forecast production/coverage diagnostics. Accuracy and actual-based eligibility belong to updated 07.")
